In [ ]:
# ============================================================
# A1 消融实验：CAMELYON16，仅自环空间连接
# 模型：一层GAT + 原关键实例聚合器
# 监督：0.5包级CE + 0.5最大实例CE
# 相对A0：保留全部patch，仅移除不同patch之间的空间连接
# ============================================================

import json
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# 使用项目原有模型类，不重新实现聚合器
from wsi.nets.dasmil import (
    GNNModel,
    FCLayer,
    BClassifier,
    MILNet,
)


class MyModel(nn.Module):
    def __init__(self, gnn, mil):
        super().__init__()
        self.gnn = gnn
        self.mil = mil

    def forward(self, features, edges):
        features = self.gnn(features, edges)
        return self.mil(features)


def get_selfloop_data(path, device):
    # 自己生成的可信特征文件
    data = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    features = data["features"].to(device)
    label = torch.as_tensor(
        data["label"],
        dtype=torch.long,
        device=device,
    ).reshape(-1)

    assert features.ndim == 2
    assert features.shape[0] > 0
    assert features.shape[1] == 384
    assert torch.isfinite(features).all()
    assert label.numel() == 1
    assert int(label.item()) in (0, 1)

    # 保留全部节点，只构造节点到自身的边
    nodes = torch.arange(
        features.shape[0],
        device=device,
        dtype=torch.long,
    )
    self_edges = torch.stack([nodes, nodes], dim=0)

    return features, self_edges, label


print("A1原模型类及仅自环数据读取函数已准备完成")

In [ ]:
# ============================================================
# A1 消融实验训练：仅自环 + GAT1 + 原聚合器 + 联合CE
# 数据集：CAMELYON16
# 划分：沿用A0的216训练 / 54验证，不重新划分
# 优化：Adam，lr=3e-4，最多100轮，patience=10
# 选模：验证集包级CE最低
# 保存：model/MIL/Ablation_NEW/CAMELYON16/
#       A1_SelfLoop_Joint_patience10/
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA不可用，请先确认GPU和当前Notebook内核。")

device = torch.device("cuda:0")

project_dir = Path("/data/code/WSI/WSI_pretrain")
data_dir = Path(
    "/data/database/CAMELYON16/pkl_data/training/20xbags/"
)

a0_dir = (
    project_dir
    / "model/MIL/compare/Camelyon/Ours/Ours_patience10"
)

# 优先读取A0自身保存的划分，再查找公共划分
split_candidates = [
    a0_dir / "split_on_CAMELYON16.json",
    a0_dir / "split.json",
    project_dir / (
        "model/MIL/compare/Camelyon/"
        "patience10_split_on_CAMELYON16.json"
    ),
    project_dir / (
        "model/MIL/compare/Camelyon/patience10_split.json"
    ),
]

split_source = next(
    (path for path in split_candidates if path.is_file()),
    None,
)

if split_source is None:
    raise FileNotFoundError(
        "没有找到A0或公共划分，请核对路径；本实验不会重新划分。"
    )

split = json.loads(
    split_source.read_text(encoding="utf-8")
)
train_files = split["train"]
val_files = split["val"]

assert len(train_files) == 216
assert len(val_files) == 54
assert len(set(train_files)) == 216
assert len(set(val_files)) == 54
assert set(train_files).isdisjoint(val_files)

if Path(split["data_path"]).resolve() != data_dir.resolve():
    raise ValueError("划分记录的数据路径与当前特征目录不一致。")

current_files = {
    path.name
    for path in data_dir.glob("*.pth")
    if path.is_file()
}
if set(train_files + val_files) != current_files:
    raise ValueError("特征文件名单与A0划分不一致，请检查数据。")

save_path = (
    project_dir
    / "model/MIL/Ablation_NEW/CAMELYON16/"
      "A1_SelfLoop_Joint_patience10"
)
save_path.mkdir(parents=True, exist_ok=True)

# 保留已有运行，避免意外覆盖
for filename in (
    "model.pth",
    "config.json",
    "split_on_CAMELYON16.json",
    "log.log",
    "training_summary.json",
):
    if (save_path / filename).exists():
        raise FileExistsError(
            f"已有实验文件，请先检查或备份：{save_path / filename}"
        )

# 与A0保持相同初始化顺序、类和默认参数
# 不额外固定随机种子，与A0的单次运行设置一致
class_num = 2

model_gnn = GNNModel(
    384, 256, 256, num_layers=1
)
model_iclassifier = FCLayer(256, class_num)
model_bclassifier = BClassifier(256, class_num)
model_mil = MILNet(
    model_iclassifier,
    model_bclassifier,
)
model = MyModel(model_gnn, model_mil).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4
)

config = {
    "experiment": "A1_SelfLoop_Joint_patience10",
    "dataset": "CAMELYON16",
    "reference_run": str(a0_dir),
    "split_source": str(split_source),
    "data_path": str(data_dir),
    "train_count": len(train_files),
    "val_count": len(val_files),
    "feature_dim": 384,
    "class_num": class_num,
    "graph_encoder": "GNNModel(384,256,256,num_layers=1)",
    "aggregation_head": "BClassifier(256,2)",
    "graph_edges": "self-loops only; all patches retained",
    "training_loss": "0.5 bag CE + 0.5 max-instance CE",
    "optimizer": "Adam",
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "patience": 10,
    "early_stopping_logic": (
        "reset to 10 after improvement, then decrement every epoch"
    ),
    "consecutive_non_improvements_to_stop": 9,
    "checkpoint_selection": "minimum validation bag CE",
    "training_seed": None,
    "train_order": "fixed saved split order; no reshuffling",
    "test_requirement": "use self-loop edges at inference too",
}

with (save_path / "config.json").open(
    "x", encoding="utf-8"
) as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

(save_path / "split_on_CAMELYON16.json").write_bytes(
    split_source.read_bytes()
)

logger = logging.getLogger("A1_SelfLoop_CAMELYON16")
logger.setLevel(logging.INFO)
logger.propagate = False

for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

handler = logging.FileHandler(
    save_path / "log.log",
    mode="a",
    encoding="utf-8",
)
handler.setFormatter(
    logging.Formatter("%(asctime)s - %(message)s")
)
logger.addHandler(handler)
logger.info(json.dumps(config, ensure_ascii=False))

print("实验：A1，仅自环＋GAT1＋原聚合器＋联合监督", flush=True)
print("划分来源：", split_source, flush=True)
print("训练 / 验证：", len(train_files), "/", len(val_files))
print("保存目录：", save_path, flush=True)

best_val_ce = float("inf")
best_epoch = None
patience = 10
remaining = patience

try:
    for epoch in range(100):
        model.train()
        train_losses = []

        for filename in tqdm(
            train_files,
            desc=f"A1 Epoch {epoch} Train",
        ):
            features, self_edges, target = get_selfloop_data(
                data_dir / filename, device
            )

            optimizer.zero_grad()

            instance_logits, bag_logits, _, _ = model(
                features, self_edges
            )

            assert tuple(bag_logits.shape) == (1, 2)
            assert tuple(instance_logits.shape) == (
                features.shape[0], 2
            )

            max_instance_logits = instance_logits.max(
                dim=0, keepdim=True
            ).values

            bag_loss = F.cross_entropy(bag_logits, target)
            instance_loss = F.cross_entropy(
                max_instance_logits, target
            )
            loss = 0.5 * bag_loss + 0.5 * instance_loss

            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"训练损失出现NaN或Inf：{filename}"
                )

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for filename in tqdm(
                val_files,
                desc=f"A1 Epoch {epoch} Val",
            ):
                features, self_edges, target = get_selfloop_data(
                    data_dir / filename, device
                )

                _, bag_logits, _, _ = model(
                    features, self_edges
                )

                # 与A0一致：按验证包级CE选模
                val_loss = F.cross_entropy(
                    bag_logits, target
                )

                if not torch.isfinite(val_loss).item():
                    raise FloatingPointError(
                        f"验证损失出现NaN或Inf：{filename}"
                    )

                val_losses.append(val_loss.item())

        train_ce = float(np.mean(train_losses))
        val_ce = float(np.mean(val_losses))
        improved = val_ce < best_val_ce

        if improved:
            best_val_ce = val_ce
            best_epoch = epoch
            remaining = patience

            torch.save(model, save_path / "model.pth")

        # 与A0原脚本保持相同早停计数逻辑
        remaining -= 1

        info = (
            f"epoch:{epoch}\tremaining:{remaining}\t"
            f"train:{train_ce:.6f}\tval:{val_ce:.6f}\t"
            + ("saved" if improved else "")
        )
        print(info, flush=True)
        logger.info(info)
        handler.flush()

        if remaining < 1:
            print("Early stopping", flush=True)
            logger.info("Early stopping")
            break

    summary = {
        "experiment": config["experiment"],
        "best_epoch": best_epoch,
        "best_validation_ce": best_val_ce,
        "weight_path": str(save_path / "model.pth"),
    }

    with (save_path / "training_summary.json").open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    logger.info("Training complete: %s", summary)

    print("\n训练完成")
    print("最佳轮次：", best_epoch)
    print("最低验证包级CE：", best_val_ce)
    print("权重位置：", save_path / "model.pth")

finally:
    handler.flush()
    handler.close()
    logger.removeHandler(handler)

In [ ]:
# ============================================================
# A1 消融实验测试：CAMELYON16，仅自环空间连接
# 模型：一层GAT + 原关键实例聚合器，联合监督训练
# 测试：保留全部patch，仅使用自环；包分支argmax分类
# 指标：类别1（肿瘤）的Precision、Recall、F1及总体Accuracy
# ============================================================

import json
from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

run_dir = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/"
    "Ablation_NEW/CAMELYON16/A1_SelfLoop_Joint_patience10/"
)
test_dir = Path(
    "/data/database/CAMELYON16/pkl_data/testing/20xbags/"
)

required = [
    "MyModel",
    "GNNModel",
    "FCLayer",
    "BClassifier",
    "MILNet",
    "get_selfloop_data",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(
        f"请先执行训练代码第一个单元中的定义：{missing}"
    )

for filename in (
    "model.pth",
    "config.json",
    "training_summary.json",
):
    if not (run_dir / filename).is_file():
        raise FileNotFoundError(run_dir / filename)

config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)
assert config["experiment"] == "A1_SelfLoop_Joint_patience10"
assert config["dataset"] == "CAMELYON16"
assert config["class_num"] == 2

weight_path = run_dir / "model.pth"
print("测试权重：", weight_path.resolve())

# 加载自己训练保存的完整模型
model = torch.load(
    weight_path,
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

test_files = sorted(
    path for path in test_dir.iterdir()
    if path.is_file() and path.suffix == ".pth"
)
assert len(test_files) == 129, (
    f"预期129张测试切片，实际为{len(test_files)}张"
)

labels, predictions, rows = [], [], []

with torch.no_grad():
    for path in tqdm(test_files, desc="A1 SelfLoop Test"):
        # 必须与训练一致：只使用自环
        features, self_edges, target = get_selfloop_data(
            path, device
        )

        # 核对每个节点恰好对应一条自环
        assert tuple(self_edges.shape) == (2, features.shape[0])
        expected_nodes = torch.arange(
            features.shape[0],
            device=features.device,
            dtype=torch.long,
        )
        assert torch.equal(self_edges[0], expected_nodes)
        assert torch.equal(self_edges[1], expected_nodes)

        _, bag_logits, _, _ = model(features, self_edges)

        assert tuple(bag_logits.shape) == (1, 2), (
            f"{path.name} 输出形状异常：{tuple(bag_logits.shape)}"
        )
        assert torch.isfinite(bag_logits).all(), (
            f"{path.name} 输出包含NaN或Inf"
        )

        true_label = int(target.item())
        prediction = int(bag_logits.argmax(dim=1).item())
        tumor_probability = float(
            bag_logits.softmax(dim=1)[0, 1].item()
        )

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": path.name,
            "label": true_label,
            "prediction": prediction,
            "tumor_probability": tumor_probability,
            "patch_count": int(features.shape[0]),
        })

distribution = [labels.count(i) for i in (0, 1)]
assert distribution == [81, 48], (
    f"测试集标签分布不一致：{distribution}"
)

result = {
    "model": "CAMELYON16_A1_SelfLoop_Joint_patience10",
    "weight_path": str(weight_path.resolve()),
    "test_data_path": str(test_dir.resolve()),
    "graph_edges": "self-loops only; all patches retained",
    "n": len(labels),
    "average": "binary",
    "positive_label": 1,
    "precision": float(precision_score(
        labels, predictions,
        pos_label=1,
        zero_division=0,
    )),
    "accuracy": float(accuracy_score(labels, predictions)),
    "recall": float(recall_score(
        labels, predictions,
        pos_label=1,
        zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions,
        pos_label=1,
        zero_division=0,
    )),
    "confusion_matrix_labels_0_1": confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别；顺序为0、1")
print(np.array(result["confusion_matrix_labels_0_1"]))

# 重复运行会更新测试JSON，不修改权重和训练记录
with (run_dir / "test_metrics.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

with (run_dir / "test_predictions.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

# 将成绩追加到训练日志
log_path = run_dir / "log.log"
with log_path.open("a", encoding="utf-8") as f:
    f.write("\n" + summary + "\n")
    f.write(
        "Confusion matrix [0, 1]:\n"
        + str(result["confusion_matrix_labels_0_1"])
        + "\n"
    )

assert summary in log_path.read_text(encoding="utf-8")
print("测试结果已保存至：", run_dir.resolve())
print("测试成绩已写入日志：", log_path.resolve())

In [ ]:
# ============================================================
# A2 消融实验：CAMELYON16，GAT1 + DSMIL聚合头
# 空间连接：八邻域，GAT内部保留自环
# 监督：0.5包级CE + 0.5最大实例CE
# 相对A0：只替换聚合头，保留原GAT及实例分类器
# DSMIL头：非线性query，value直接使用GAT特征
# ============================================================

import json
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

from wsi.nets.dasmil import GNNModel, FCLayer, MILNet


# 使用独立类名，避免覆盖A0/A1的BClassifier
class A2DSMILBagClassifier(nn.Module):
    def __init__(self, input_size=256, class_num=2):
        super().__init__()

        self.q = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.Tanh(),
        )

        # 对应DSMIL的passing_v=False
        self.v = nn.Identity()

        self.fcc = nn.Conv1d(
            class_num,
            class_num,
            kernel_size=input_size,
        )

    def forward(self, feats, instance_logits):
        values = self.v(feats)
        queries = self.q(feats)

        # 每类选取最高分实例作为参考
        _, indices = torch.sort(
            instance_logits,
            dim=0,
            descending=True,
        )
        reference_features = feats.index_select(
            0, indices[0]
        )
        reference_queries = self.q(reference_features)

        attention = queries @ reference_queries.transpose(0, 1)
        scale = queries.new_tensor(queries.shape[1]).sqrt()
        attention = F.softmax(attention / scale, dim=0)

        bag_features = (
            attention.transpose(0, 1) @ values
        ).unsqueeze(0)

        bag_logits = self.fcc(bag_features).reshape(1, -1)

        return bag_logits, attention, bag_features


class A2GATDSMIL(nn.Module):
    def __init__(self, gnn, mil):
        super().__init__()
        self.gnn = gnn
        self.mil = mil

    def forward(self, features, edges):
        return self.mil(self.gnn(features, edges))


def get_a2_data(path, device):
    data = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    features = data["features"].to(device)
    label = torch.as_tensor(
        data["label"],
        dtype=torch.long,
        device=device,
    ).reshape(-1)

    coords = [
        tuple(map(int, xy))
        for xy in data["rects"]
    ]
    coordinate_index = {
        xy: i for i, xy in enumerate(coords)
    }

    assert features.ndim == 2
    assert features.shape[1] == 384
    assert features.shape[0] > 0
    assert features.shape[0] == len(coords)
    assert len(coordinate_index) == len(coords), "存在重复坐标"
    assert torch.isfinite(features).all()
    assert label.numel() == 1
    assert int(label.item()) in (0, 1)

    stride = int(data["patch_size"])
    assert stride > 0

    offsets = [
        (stride, -stride),
        (stride, 0),
        (stride, stride),
        (0, -stride),
        (0, stride),
        (-stride, -stride),
        (-stride, 0),
        (-stride, stride),
    ]

    edges = []
    for i, (x, y) in enumerate(coords):
        for dx, dy in offsets:
            j = coordinate_index.get((x + dx, y + dy))
            if j is not None:
                edges.append([i, j])

    # 无外部边时仍保留节点，由GAT内部添加自环
    if edges:
        edge_index = torch.tensor(
            edges, dtype=torch.long, device=device
        ).t().contiguous()
    else:
        edge_index = torch.empty(
            (2, 0), dtype=torch.long, device=device
        )

    return features, edge_index, label


print("A2模型与八邻域数据读取函数已准备完成")

In [ ]:
# ============================================================
# A2训练：八邻域GAT1 + DSMIL头 + 联合CE
# 划分：沿用A0，216训练 / 54验证
# 优化：Adam，lr=3e-4，最多100轮，patience=10
# 选模：最低验证包级CE
# 保存：Ablation_NEW/CAMELYON16/A2_GAT_DSMIL_Joint_patience10
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA不可用，请检查GPU和Notebook内核。")

device = torch.device("cuda:0")

project_dir = Path("/data/code/WSI/WSI_pretrain")
data_dir = Path(
    "/data/database/CAMELYON16/pkl_data/training/20xbags/"
)
a0_dir = (
    project_dir
    / "model/MIL/compare/Camelyon/Ours/Ours_patience10"
)

split_candidates = [
    a0_dir / "split_on_CAMELYON16.json",
    a0_dir / "split.json",
    project_dir / (
        "model/MIL/compare/Camelyon/"
        "patience10_split_on_CAMELYON16.json"
    ),
    project_dir / (
        "model/MIL/compare/Camelyon/patience10_split.json"
    ),
]

split_source = next(
    (p for p in split_candidates if p.is_file()),
    None,
)
if split_source is None:
    raise FileNotFoundError("没有找到A0或公共划分，不重新随机划分。")

split = json.loads(
    split_source.read_text(encoding="utf-8")
)
train_files = split["train"]
val_files = split["val"]

assert len(train_files) == len(set(train_files)) == 216
assert len(val_files) == len(set(val_files)) == 54
assert set(train_files).isdisjoint(val_files)

if Path(split["data_path"]).resolve() != data_dir.resolve():
    raise ValueError("划分记录与当前数据路径不一致。")

current_files = {
    p.name for p in data_dir.glob("*.pth")
    if p.is_file()
}
if current_files != set(train_files + val_files):
    raise ValueError("当前特征文件名单与A0划分不一致。")

a2_save_path = (
    project_dir
    / "model/MIL/Ablation_NEW/CAMELYON16/"
      "A2_GAT_DSMIL_Joint_patience10"
)
a2_save_path.mkdir(parents=True, exist_ok=True)

for filename in (
    "model.pth",
    "config.json",
    "split_on_CAMELYON16.json",
    "log.log",
    "training_summary.json",
):
    if (a2_save_path / filename).exists():
        raise FileExistsError(
            f"已有实验文件，请先检查或备份：{a2_save_path / filename}"
        )

# 与A0保持相同GAT和实例分类器
model_gnn = GNNModel(384, 256, 256, num_layers=1)
model_iclassifier = FCLayer(256, 2)

# 仅替换包聚合头
model_bclassifier = A2DSMILBagClassifier(256, 2)
model_mil = MILNet(model_iclassifier, model_bclassifier)
model = A2GATDSMIL(model_gnn, model_mil).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4
)

config = {
    "experiment": "A2_GAT_DSMIL_Joint_patience10",
    "dataset": "CAMELYON16",
    "class_num": 2,
    "feature_dim": 384,
    "reference_run": str(a0_dir),
    "split_source": str(split_source),
    "data_path": str(data_dir),
    "graph_encoder": "GNNModel(384,256,256,num_layers=1)",
    "graph_edges": "eight-neighbor; internal GAT self-loops",
    "aggregation_head": "A2DSMILBagClassifier(256,2)",
    "query": "Linear(256,128)-ReLU-Linear(128,128)-Tanh",
    "value": "Identity; passing_v=False",
    "training_loss": "0.5 bag CE + 0.5 max-instance CE",
    "optimizer": "Adam",
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "patience": 10,
    "early_stopping_logic": (
        "reset to 10 on improvement, then decrement every epoch"
    ),
    "consecutive_non_improvements_to_stop": 9,
    "checkpoint_selection": "minimum validation bag CE",
    "training_seed": None,
    "train_order": "fixed saved split order; no reshuffling",
}

with (a2_save_path / "config.json").open(
    "x", encoding="utf-8"
) as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

(a2_save_path / "split_on_CAMELYON16.json").write_bytes(
    split_source.read_bytes()
)

logger = logging.getLogger("A2_GAT_DSMIL_CAMELYON16")
logger.setLevel(logging.INFO)
logger.propagate = False

for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

handler = logging.FileHandler(
    a2_save_path / "log.log",
    mode="a",
    encoding="utf-8",
)
handler.setFormatter(
    logging.Formatter("%(asctime)s - %(message)s")
)
logger.addHandler(handler)
logger.info(json.dumps(config, ensure_ascii=False))

print("实验：A2，八邻域GAT1＋DSMIL头＋联合CE", flush=True)
print("划分来源：", split_source, flush=True)
print("保存目录：", a2_save_path, flush=True)

best_val_ce = float("inf")
best_epoch = None
remaining = 10

try:
    for epoch in range(100):
        model.train()
        train_losses = []

        for filename in tqdm(
            train_files, desc=f"A2 Epoch {epoch} Train"
        ):
            features, edges, target = get_a2_data(
                data_dir / filename, device
            )
            optimizer.zero_grad()

            instance_logits, bag_logits, _, _ = model(
                features, edges
            )

            assert tuple(bag_logits.shape) == (1, 2)
            assert tuple(instance_logits.shape) == (
                features.shape[0], 2
            )

            max_logits = instance_logits.max(
                dim=0, keepdim=True
            ).values

            # 与A0相同的联合CE，不改成独立DSMIL的BCE配方
            loss = (
                0.5 * F.cross_entropy(bag_logits, target)
                + 0.5 * F.cross_entropy(max_logits, target)
            )

            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"非有限训练损失：{filename}"
                )

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for filename in tqdm(
                val_files, desc=f"A2 Epoch {epoch} Val"
            ):
                features, edges, target = get_a2_data(
                    data_dir / filename, device
                )
                _, bag_logits, _, _ = model(features, edges)
                loss = F.cross_entropy(bag_logits, target)

                if not torch.isfinite(loss).item():
                    raise FloatingPointError(
                        f"非有限验证损失：{filename}"
                    )

                val_losses.append(loss.item())

        train_ce = float(np.mean(train_losses))
        val_ce = float(np.mean(val_losses))
        improved = val_ce < best_val_ce

        if improved:
            best_val_ce = val_ce
            best_epoch = epoch
            remaining = 10
            torch.save(model, a2_save_path / "model.pth")

        # 与A0/A1一致的计数逻辑
        remaining -= 1

        info = (
            f"epoch:{epoch}\tremaining:{remaining}\t"
            f"train:{train_ce:.6f}\tval:{val_ce:.6f}\t"
            + ("saved" if improved else "")
        )
        print(info, flush=True)
        logger.info(info)
        handler.flush()

        if remaining < 1:
            print("Early stopping", flush=True)
            logger.info("Early stopping")
            break

    summary = {
        "experiment": config["experiment"],
        "best_epoch": best_epoch,
        "best_validation_ce": best_val_ce,
        "weight_path": str(a2_save_path / "model.pth"),
    }

    with (a2_save_path / "training_summary.json").open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    logger.info("Training complete: %s", summary)
    print("\n训练完成：", summary)

finally:
    handler.flush()
    handler.close()
    logger.removeHandler(handler)

In [ ]:
# ============================================================
# A2消融实验测试：CAMELYON16，八邻域GAT1 + DSMIL头
# 加载最佳权重，使用包分支预测
# 指标：肿瘤类别1的Precision、Recall、F1及总体Accuracy
# 测试结果写入本次目录，并追加到log.log
# ============================================================

from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

run_dir = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/"
    "Ablation_NEW/CAMELYON16/A2_GAT_DSMIL_Joint_patience10/"
)
test_dir = Path(
    "/data/database/CAMELYON16/pkl_data/testing/20xbags/"
)

# 若重启过内核，先重新运行单元1
for filename in (
    "model.pth",
    "config.json",
    "training_summary.json",
):
    if not (run_dir / filename).is_file():
        raise FileNotFoundError(run_dir / filename)

test_config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)
assert test_config["experiment"] == "A2_GAT_DSMIL_Joint_patience10"

model = torch.load(
    run_dir / "model.pth",
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

test_files = sorted(
    p for p in test_dir.iterdir()
    if p.is_file() and p.suffix == ".pth"
)
assert len(test_files) == 129

labels, predictions, rows = [], [], []

with torch.no_grad():
    for path in tqdm(test_files, desc="A2 GAT DSMIL Test"):
        # 与训练一致，使用八邻域边
        features, edges, target = get_a2_data(path, device)
        _, bag_logits, _, _ = model(features, edges)

        assert tuple(bag_logits.shape) == (1, 2)
        assert torch.isfinite(bag_logits).all()

        true_label = int(target.item())
        prediction = int(bag_logits.argmax(dim=1).item())

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": path.name,
            "label": true_label,
            "prediction": prediction,
            "tumor_probability": float(
                bag_logits.softmax(dim=1)[0, 1].item()
            ),
        })

assert [labels.count(i) for i in (0, 1)] == [81, 48]

result = {
    "model": "CAMELYON16_A2_GAT_DSMIL_Joint_patience10",
    "weight_path": str((run_dir / "model.pth").resolve()),
    "test_data_path": str(test_dir.resolve()),
    "n": len(labels),
    "average": "binary",
    "positive_label": 1,
    "precision": float(precision_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "accuracy": float(accuracy_score(labels, predictions)),
    "recall": float(recall_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "confusion_matrix_labels_0_1": confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别；顺序0、1")
print(np.array(result["confusion_matrix_labels_0_1"]))

# 重复测试会更新JSON，不修改权重
for filename, content in (
    ("test_metrics.json", result),
    ("test_predictions.json", rows),
):
    with (run_dir / filename).open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

log_path = run_dir / "log.log"
with log_path.open("a", encoding="utf-8") as f:
    f.write("\n" + summary + "\n")
    f.write(
        "Confusion matrix [0, 1]:\n"
        + str(result["confusion_matrix_labels_0_1"])
        + "\n"
    )

assert summary in log_path.read_text(encoding="utf-8")
print("测试结果和日志已保存至：", run_dir.resolve())

In [ ]:
# ============================================================
# A3 消融实验：CAMELYON16，常规自注意力聚合配置
# 模型：八邻域GAT1 + 原BClassifier_noselect的分类计算
# 监督：0.5包级CE + 0.5最大实例CE
# 读出：原实现使用自注意力更新后的第一个位置
# 优化：只计算该位置所需的注意力列，不构造N×N矩阵
# ============================================================

import json
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

from wsi.nets.dasmil import GNNModel, FCLayer, MILNet


class A3BClassifierNoSelect(nn.Module):
    def __init__(self, input_size=256, class_num=2):
        super().__init__()

        # 与原BClassifier_noselect相同的参数结构
        self.lin = nn.Sequential(
            nn.Linear(input_size, input_size),
            nn.ReLU(),
        )
        self.q = nn.Sequential(
            nn.Dropout(0.0),
            nn.Linear(input_size, 128),
        )
        self.k = nn.Sequential(
            nn.Dropout(0.0),
            nn.Linear(input_size, 128),
        )
        self.v = nn.Sequential(
            nn.Dropout(0.0),
            nn.Linear(input_size, input_size),
        )
        self.fcc = nn.Sequential(
            nn.Linear(input_size, class_num)
        )

    def forward(self, feats, instance_logits):
        # 此包分支不使用实例分数进行参考选择。
        # 实例分数仍由MILNet返回，用于联合监督。
        feats = self.lin(feats)
        queries = self.q(feats)       # [N,128]
        keys = self.k(feats)          # [N,128]
        values = self.v(feats)        # [N,256]

        # 原实现：
        # A = softmax(Q @ K.T / sqrt(128), dim=0)
        # B = A.T @ V
        # logits = fcc(B[0])
        #
        # 仅计算B[0]依赖的A[:,0]，避免N×N矩阵。
        scores = queries @ keys[0:1].transpose(0, 1)
        scale = queries.new_tensor(queries.shape[1]).sqrt()
        attention = F.softmax(scores / scale, dim=0)  # [N,1]

        bag_vector = attention.transpose(0, 1) @ values
        bag_logits = self.fcc(bag_vector)              # [1,2]

        return bag_logits, attention, bag_vector.unsqueeze(0)


class A3GATNoSelect(nn.Module):
    def __init__(self, gnn, mil):
        super().__init__()
        self.gnn = gnn
        self.mil = mil

    def forward(self, features, edges):
        return self.mil(self.gnn(features, edges))


# 沿用A2的八邻域读取函数，不改变patch顺序和构图规则
if "get_a2_data" not in globals():
    raise NameError(
        "请先运行上一实验A2的单元1，定义get_a2_data。"
    )

get_a3_data = get_a2_data

print("A3模型已定义，使用与A2相同的八邻域数据读取")

In [ ]:
# ============================================================
# A3训练：八邻域GAT1 + noselect聚合配置 + 联合CE
# 划分：沿用A0，216训练 / 54验证
# 优化：Adam，lr=3e-4，最多100轮，patience=10
# 选模：最低验证包级CE
# 保存：Ablation_NEW/CAMELYON16/A3_GAT_NoSelect_Joint_patience10
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA不可用，请检查GPU和Notebook内核。")

device = torch.device("cuda:0")

project_dir = Path("/data/code/WSI/WSI_pretrain")
data_dir = Path(
    "/data/database/CAMELYON16/pkl_data/training/20xbags/"
)
a0_dir = (
    project_dir
    / "model/MIL/compare/Camelyon/Ours/Ours_patience10"
)

split_candidates = [
    a0_dir / "split_on_CAMELYON16.json",
    a0_dir / "split.json",
    project_dir / (
        "model/MIL/compare/Camelyon/"
        "patience10_split_on_CAMELYON16.json"
    ),
    project_dir / (
        "model/MIL/compare/Camelyon/patience10_split.json"
    ),
]

split_source = next(
    (p for p in split_candidates if p.is_file()),
    None,
)
if split_source is None:
    raise FileNotFoundError("未找到A0或公共划分，不重新随机划分。")

split = json.loads(
    split_source.read_text(encoding="utf-8")
)
train_files = split["train"]
val_files = split["val"]

assert len(train_files) == len(set(train_files)) == 216
assert len(val_files) == len(set(val_files)) == 54
assert set(train_files).isdisjoint(val_files)

if Path(split["data_path"]).resolve() != data_dir.resolve():
    raise ValueError("划分记录与当前数据目录不一致。")

current_files = {
    p.name for p in data_dir.glob("*.pth")
    if p.is_file()
}
if current_files != set(train_files + val_files):
    raise ValueError("当前文件名单与A0划分不一致。")

a3_save_path = (
    project_dir
    / "model/MIL/Ablation_NEW/CAMELYON16/"
      "A3_GAT_NoSelect_Joint_patience10"
)
a3_save_path.mkdir(parents=True, exist_ok=True)

for filename in (
    "model.pth",
    "config.json",
    "split_on_CAMELYON16.json",
    "log.log",
    "training_summary.json",
):
    if (a3_save_path / filename).exists():
        raise FileExistsError(
            f"已有实验文件，请先检查或备份：{a3_save_path / filename}"
        )

# 与A0保持相同GAT和实例分类器
model_gnn = GNNModel(384, 256, 256, num_layers=1)
model_iclassifier = FCLayer(256, 2)
model_bclassifier = A3BClassifierNoSelect(256, 2)
model_mil = MILNet(model_iclassifier, model_bclassifier)
model = A3GATNoSelect(model_gnn, model_mil).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4
)

config = {
    "experiment": "A3_GAT_NoSelect_Joint_patience10",
    "dataset": "CAMELYON16",
    "class_num": 2,
    "feature_dim": 384,
    "reference_run": str(a0_dir),
    "split_source": str(split_source),
    "data_path": str(data_dir),
    "graph_encoder": "GNNModel(384,256,256,num_layers=1)",
    "graph_edges": "eight-neighbor; internal GAT self-loops",
    "aggregation_head": "original BClassifier_noselect classification",
    "readout": "linear classifier on updated first position",
    "attention_implementation": (
        "compute only column 0 of original dim-0 normalized attention"
    ),
    "instance_branch": "FCLayer(256,2), retained for joint supervision",
    "training_loss": "0.5 bag CE + 0.5 max-instance CE",
    "optimizer": "Adam",
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "patience": 10,
    "early_stopping_logic": (
        "reset to 10 on improvement, then decrement every epoch"
    ),
    "consecutive_non_improvements_to_stop": 9,
    "checkpoint_selection": "minimum validation bag CE",
    "training_seed": None,
    "train_order": "fixed saved split order; no reshuffling",
}

with (a3_save_path / "config.json").open(
    "x", encoding="utf-8"
) as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

(a3_save_path / "split_on_CAMELYON16.json").write_bytes(
    split_source.read_bytes()
)

logger = logging.getLogger("A3_GAT_NoSelect_CAMELYON16")
logger.setLevel(logging.INFO)
logger.propagate = False

for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

handler = logging.FileHandler(
    a3_save_path / "log.log",
    mode="a",
    encoding="utf-8",
)
handler.setFormatter(
    logging.Formatter("%(asctime)s - %(message)s")
)
logger.addHandler(handler)
logger.info(json.dumps(config, ensure_ascii=False))

print("实验：A3，八邻域GAT1＋noselect＋联合CE", flush=True)
print("划分来源：", split_source, flush=True)
print("保存目录：", a3_save_path, flush=True)

best_val_ce = float("inf")
best_epoch = None
remaining = 10

try:
    for epoch in range(100):
        model.train()
        train_losses = []

        for filename in tqdm(
            train_files, desc=f"A3 Epoch {epoch} Train"
        ):
            features, edges, target = get_a3_data(
                data_dir / filename, device
            )
            optimizer.zero_grad()

            instance_logits, bag_logits, _, _ = model(
                features, edges
            )

            assert tuple(bag_logits.shape) == (1, 2)
            assert tuple(instance_logits.shape) == (
                features.shape[0], 2
            )

            max_logits = instance_logits.max(
                dim=0, keepdim=True
            ).values

            loss = (
                0.5 * F.cross_entropy(bag_logits, target)
                + 0.5 * F.cross_entropy(max_logits, target)
            )

            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"非有限训练损失：{filename}"
                )

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for filename in tqdm(
                val_files, desc=f"A3 Epoch {epoch} Val"
            ):
                features, edges, target = get_a3_data(
                    data_dir / filename, device
                )
                _, bag_logits, _, _ = model(features, edges)
                loss = F.cross_entropy(bag_logits, target)

                if not torch.isfinite(loss).item():
                    raise FloatingPointError(
                        f"非有限验证损失：{filename}"
                    )

                val_losses.append(loss.item())

        train_ce = float(np.mean(train_losses))
        val_ce = float(np.mean(val_losses))
        improved = val_ce < best_val_ce

        if improved:
            best_val_ce = val_ce
            best_epoch = epoch
            remaining = 10
            torch.save(model, a3_save_path / "model.pth")

        remaining -= 1

        info = (
            f"epoch:{epoch}\tremaining:{remaining}\t"
            f"train:{train_ce:.6f}\tval:{val_ce:.6f}\t"
            + ("saved" if improved else "")
        )
        print(info, flush=True)
        logger.info(info)
        handler.flush()

        if remaining < 1:
            print("Early stopping", flush=True)
            logger.info("Early stopping")
            break

    summary = {
        "experiment": config["experiment"],
        "best_epoch": best_epoch,
        "best_validation_ce": best_val_ce,
        "weight_path": str(a3_save_path / "model.pth"),
    }

    with (a3_save_path / "training_summary.json").open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    logger.info("Training complete: %s", summary)
    print("\n训练完成：", summary)

finally:
    handler.flush()
    handler.close()
    logger.removeHandler(handler)

In [ ]:
# ============================================================
# A3消融实验测试：CAMELYON16，八邻域GAT1 + noselect
# 预测：最佳权重，包分支argmax
# 指标：类别1（肿瘤）的Pre、Rec、F1，以及总体Acc
# ============================================================

from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

run_dir = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/"
    "Ablation_NEW/CAMELYON16/A3_GAT_NoSelect_Joint_patience10/"
)
test_dir = Path(
    "/data/database/CAMELYON16/pkl_data/testing/20xbags/"
)

# 若重启内核，先重新执行A2单元1及本实验单元1
for filename in (
    "model.pth",
    "config.json",
    "training_summary.json",
):
    if not (run_dir / filename).is_file():
        raise FileNotFoundError(run_dir / filename)

test_config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)
assert test_config["experiment"] == "A3_GAT_NoSelect_Joint_patience10"

model = torch.load(
    run_dir / "model.pth",
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

test_files = sorted(
    p for p in test_dir.iterdir()
    if p.is_file() and p.suffix == ".pth"
)
assert len(test_files) == 129

labels, predictions, rows = [], [], []

with torch.no_grad():
    for path in tqdm(test_files, desc="A3 GAT NoSelect Test"):
        features, edges, target = get_a3_data(path, device)

        # 保持原patch顺序；原noselect使用第一个位置读出
        _, bag_logits, _, _ = model(features, edges)

        assert tuple(bag_logits.shape) == (1, 2)
        assert torch.isfinite(bag_logits).all()

        true_label = int(target.item())
        prediction = int(bag_logits.argmax(dim=1).item())

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": path.name,
            "label": true_label,
            "prediction": prediction,
            "tumor_probability": float(
                bag_logits.softmax(dim=1)[0, 1].item()
            ),
        })

assert [labels.count(i) for i in (0, 1)] == [81, 48]

result = {
    "model": "CAMELYON16_A3_GAT_NoSelect_Joint_patience10",
    "weight_path": str((run_dir / "model.pth").resolve()),
    "test_data_path": str(test_dir.resolve()),
    "n": len(labels),
    "average": "binary",
    "positive_label": 1,
    "precision": float(precision_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "accuracy": float(accuracy_score(labels, predictions)),
    "recall": float(recall_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "confusion_matrix_labels_0_1": confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别；顺序0、1")
print(np.array(result["confusion_matrix_labels_0_1"]))

for filename, content in (
    ("test_metrics.json", result),
    ("test_predictions.json", rows),
):
    with (run_dir / filename).open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

log_path = run_dir / "log.log"
with log_path.open("a", encoding="utf-8") as f:
    f.write("\n" + summary + "\n")
    f.write(
        "Confusion matrix [0, 1]:\n"
        + str(result["confusion_matrix_labels_0_1"])
        + "\n"
    )

assert summary in log_path.read_text(encoding="utf-8")
print("测试结果和日志已保存至：", run_dir.resolve())

In [ ]:
# ============================================================
# B1 消融实验：CAMELYON16，两层GAT
# 模型：八邻域GAT2 + 原关键实例聚合器
# 监督：0.5包级CE + 0.5最大实例CE
# 相对A0：GAT层数由1改为2，采用原GNNModel多层结构
# 划分：沿用A0的216训练 / 54验证
# 优化：Adam，lr=3e-4，最多100轮，patience=10
# 选模：最低验证包级CE
# ============================================================

import json
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# 使用原模型，避免误用前面A2/A3定义的聚合头
from wsi.nets.dasmil import (
    GNNModel,
    FCLayer,
    BClassifier,
    MILNet,
)

if "get_a2_data" not in globals():
    raise NameError(
        "请先执行A2的第一个单元，定义八邻域读取函数get_a2_data。"
    )

get_b1_data = get_a2_data


class B1GAT2Model(nn.Module):
    def __init__(self, gnn, mil):
        super().__init__()
        self.gnn = gnn
        self.mil = mil

    def forward(self, features, edges):
        return self.mil(self.gnn(features, edges))


if not torch.cuda.is_available():
    raise RuntimeError("CUDA不可用，请检查GPU和Notebook内核。")

device = torch.device("cuda:0")

project_dir = Path("/data/code/WSI/WSI_pretrain")
data_dir = Path(
    "/data/database/CAMELYON16/pkl_data/training/20xbags/"
)
a0_dir = (
    project_dir
    / "model/MIL/compare/Camelyon/Ours/Ours_patience10"
)

split_candidates = [
    a0_dir / "split_on_CAMELYON16.json",
    a0_dir / "split.json",
    project_dir / (
        "model/MIL/compare/Camelyon/"
        "patience10_split_on_CAMELYON16.json"
    ),
    project_dir / (
        "model/MIL/compare/Camelyon/patience10_split.json"
    ),
]

split_source = next(
    (p for p in split_candidates if p.is_file()),
    None,
)
if split_source is None:
    raise FileNotFoundError("未找到A0或公共划分，不重新随机划分。")

split = json.loads(
    split_source.read_text(encoding="utf-8")
)
train_files = split["train"]
val_files = split["val"]

assert len(train_files) == len(set(train_files)) == 216
assert len(val_files) == len(set(val_files)) == 54
assert set(train_files).isdisjoint(val_files)

if Path(split["data_path"]).resolve() != data_dir.resolve():
    raise ValueError("划分记录与当前数据目录不一致。")

current_files = {
    p.name for p in data_dir.glob("*.pth")
    if p.is_file()
}
if current_files != set(train_files + val_files):
    raise ValueError("当前特征文件名单与A0划分不一致。")

b1_save_path = (
    project_dir
    / "model/MIL/Ablation_NEW/CAMELYON16/"
      "B1_GAT2_Joint_patience10"
)
b1_save_path.mkdir(parents=True, exist_ok=True)

for filename in (
    "model.pth",
    "config.json",
    "split_on_CAMELYON16.json",
    "log.log",
    "training_summary.json",
):
    if (b1_save_path / filename).exists():
        raise FileExistsError(
            f"已有实验文件，请先检查或备份：{b1_save_path / filename}"
        )

# 相对A0：仅修改num_layers=2，采用原类默认结构
model_gnn = GNNModel(
    384, 256, 256, num_layers=2
)
model_iclassifier = FCLayer(256, 2)
model_bclassifier = BClassifier(256, 2)
model_mil = MILNet(model_iclassifier, model_bclassifier)
model = B1GAT2Model(model_gnn, model_mil).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4
)

# 保存原类实际生成的网络结构，便于核对各层配置
print("本次图编码器：")
print(model_gnn)

config = {
    "experiment": "B1_GAT2_Joint_patience10",
    "dataset": "CAMELYON16",
    "class_num": 2,
    "feature_dim": 384,
    "reference_run": str(a0_dir),
    "split_source": str(split_source),
    "data_path": str(data_dir),
    "graph_encoder": "GNNModel(384,256,256,num_layers=2)",
    "graph_structure": str(model_gnn),
    "graph_edges": "eight-neighbor; internal GAT self-loops",
    "aggregation_head": "original BClassifier(256,2)",
    "training_loss": "0.5 bag CE + 0.5 max-instance CE",
    "optimizer": "Adam",
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "patience": 10,
    "early_stopping_logic": (
        "reset to 10 on improvement, then decrement every epoch"
    ),
    "consecutive_non_improvements_to_stop": 9,
    "checkpoint_selection": "minimum validation bag CE",
    "training_seed": None,
    "train_order": "fixed saved split order; no reshuffling",
}

with (b1_save_path / "config.json").open(
    "x", encoding="utf-8"
) as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

(b1_save_path / "split_on_CAMELYON16.json").write_bytes(
    split_source.read_bytes()
)

logger = logging.getLogger("B1_GAT2_CAMELYON16")
logger.setLevel(logging.INFO)
logger.propagate = False

for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

handler = logging.FileHandler(
    b1_save_path / "log.log",
    mode="a",
    encoding="utf-8",
)
handler.setFormatter(
    logging.Formatter("%(asctime)s - %(message)s")
)
logger.addHandler(handler)
logger.info(json.dumps(config, ensure_ascii=False))

print("实验：B1，八邻域GAT2＋原聚合器＋联合CE", flush=True)
print("划分来源：", split_source, flush=True)
print("保存目录：", b1_save_path, flush=True)

best_val_ce = float("inf")
best_epoch = None
remaining = 10

try:
    for epoch in range(100):
        model.train()
        train_losses = []

        for filename in tqdm(
            train_files, desc=f"B1 GAT2 Epoch {epoch} Train"
        ):
            features, edges, target = get_b1_data(
                data_dir / filename, device
            )
            optimizer.zero_grad()

            instance_logits, bag_logits, _, _ = model(
                features, edges
            )

            assert tuple(bag_logits.shape) == (1, 2)
            assert tuple(instance_logits.shape) == (
                features.shape[0], 2
            )

            max_logits = instance_logits.max(
                dim=0, keepdim=True
            ).values

            loss = (
                0.5 * F.cross_entropy(bag_logits, target)
                + 0.5 * F.cross_entropy(max_logits, target)
            )

            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"非有限训练损失：{filename}"
                )

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for filename in tqdm(
                val_files, desc=f"B1 GAT2 Epoch {epoch} Val"
            ):
                features, edges, target = get_b1_data(
                    data_dir / filename, device
                )
                _, bag_logits, _, _ = model(features, edges)

                # 与A0相同，使用验证包级CE选模
                loss = F.cross_entropy(bag_logits, target)

                if not torch.isfinite(loss).item():
                    raise FloatingPointError(
                        f"非有限验证损失：{filename}"
                    )

                val_losses.append(loss.item())

        train_ce = float(np.mean(train_losses))
        val_ce = float(np.mean(val_losses))
        improved = val_ce < best_val_ce

        if improved:
            best_val_ce = val_ce
            best_epoch = epoch
            remaining = 10
            torch.save(model, b1_save_path / "model.pth")

        remaining -= 1

        info = (
            f"epoch:{epoch}\tremaining:{remaining}\t"
            f"train:{train_ce:.6f}\tval:{val_ce:.6f}\t"
            + ("saved" if improved else "")
        )
        print(info, flush=True)
        logger.info(info)
        handler.flush()

        if remaining < 1:
            print("Early stopping", flush=True)
            logger.info("Early stopping")
            break

    summary = {
        "experiment": config["experiment"],
        "best_epoch": best_epoch,
        "best_validation_ce": best_val_ce,
        "weight_path": str(b1_save_path / "model.pth"),
    }

    with (b1_save_path / "training_summary.json").open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    logger.info("Training complete: %s", summary)
    print("\n训练完成：", summary)

finally:
    handler.flush()
    handler.close()
    logger.removeHandler(handler)

In [ ]:
# ============================================================
# B1消融实验测试：CAMELYON16，两层GAT + 原聚合器
# 输入：原八邻域图，保持patch顺序
# 预测：最佳权重，包分支argmax
# 指标：肿瘤类别1的Pre、Rec、F1，以及总体Acc
# ============================================================

from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

run_dir = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/"
    "Ablation_NEW/CAMELYON16/B1_GAT2_Joint_patience10/"
)
test_dir = Path(
    "/data/database/CAMELYON16/pkl_data/testing/20xbags/"
)

# 重启过内核时：
# 先恢复原模型类、B1GAT2Model以及get_a2_data的定义。
if "get_a2_data" not in globals():
    raise NameError("请先恢复A2单元1中的get_a2_data定义。")

if "B1GAT2Model" not in globals():
    raise NameError("请先恢复训练单元中的B1GAT2Model类定义。")

for filename in (
    "model.pth",
    "config.json",
    "training_summary.json",
):
    if not (run_dir / filename).is_file():
        raise FileNotFoundError(run_dir / filename)

test_config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)
assert test_config["experiment"] == "B1_GAT2_Joint_patience10"

model = torch.load(
    run_dir / "model.pth",
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

test_files = sorted(
    p for p in test_dir.iterdir()
    if p.is_file() and p.suffix == ".pth"
)
assert len(test_files) == 129

labels, predictions, rows = [], [], []

with torch.no_grad():
    for path in tqdm(test_files, desc="B1 GAT2 Test"):
        features, edges, target = get_a2_data(path, device)
        _, bag_logits, _, _ = model(features, edges)

        assert tuple(bag_logits.shape) == (1, 2)
        assert torch.isfinite(bag_logits).all()

        true_label = int(target.item())
        prediction = int(bag_logits.argmax(dim=1).item())

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": path.name,
            "label": true_label,
            "prediction": prediction,
            "tumor_probability": float(
                bag_logits.softmax(dim=1)[0, 1].item()
            ),
        })

assert [labels.count(i) for i in (0, 1)] == [81, 48]

result = {
    "model": "CAMELYON16_B1_GAT2_Joint_patience10",
    "weight_path": str((run_dir / "model.pth").resolve()),
    "test_data_path": str(test_dir.resolve()),
    "n": len(labels),
    "average": "binary",
    "positive_label": 1,
    "precision": float(precision_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "accuracy": float(accuracy_score(labels, predictions)),
    "recall": float(recall_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "confusion_matrix_labels_0_1": confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别；顺序0、1")
print(np.array(result["confusion_matrix_labels_0_1"]))

for filename, content in (
    ("test_metrics.json", result),
    ("test_predictions.json", rows),
):
    with (run_dir / filename).open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

log_path = run_dir / "log.log"
with log_path.open("a", encoding="utf-8") as f:
    f.write("\n" + summary + "\n")
    f.write(
        "Confusion matrix [0, 1]:\n"
        + str(result["confusion_matrix_labels_0_1"])
        + "\n"
    )

assert summary in log_path.read_text(encoding="utf-8")
print("测试结果和日志已保存至：", run_dir.resolve())

In [ ]:
# ============================================================
# B2 消融实验：CAMELYON16，三层GAT
# 模型：八邻域GAT3 + 原关键实例聚合器
# 监督：0.5包级CE + 0.5最大实例CE
# 相对A0：采用原GNNModel的三层配置
# 划分：沿用A0的216训练 / 54验证，不重新划分
# 优化：Adam，lr=3e-4，最多100轮，patience=10
# 选模：最低验证包级CE
# ============================================================

import json
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

from wsi.nets.dasmil import (
    GNNModel,
    FCLayer,
    BClassifier,
    MILNet,
)

if "get_a2_data" not in globals():
    raise NameError(
        "请先执行A2的单元1，定义八邻域读取函数get_a2_data。"
    )


class B2GAT3Model(nn.Module):
    def __init__(self, gnn, mil):
        super().__init__()
        self.gnn = gnn
        self.mil = mil

    def forward(self, features, edges):
        return self.mil(self.gnn(features, edges))


if not torch.cuda.is_available():
    raise RuntimeError("CUDA不可用，请检查GPU和Notebook内核。")

device = torch.device("cuda:0")

project_dir = Path("/data/code/WSI/WSI_pretrain")
data_dir = Path(
    "/data/database/CAMELYON16/pkl_data/training/20xbags/"
)
a0_dir = (
    project_dir
    / "model/MIL/compare/Camelyon/Ours/Ours_patience10"
)

split_candidates = [
    a0_dir / "split_on_CAMELYON16.json",
    a0_dir / "split.json",
    project_dir / (
        "model/MIL/compare/Camelyon/"
        "patience10_split_on_CAMELYON16.json"
    ),
    project_dir / (
        "model/MIL/compare/Camelyon/patience10_split.json"
    ),
]

split_source = next(
    (p for p in split_candidates if p.is_file()),
    None,
)
if split_source is None:
    raise FileNotFoundError("未找到A0或公共划分，不重新随机划分。")

split = json.loads(
    split_source.read_text(encoding="utf-8")
)
train_files = split["train"]
val_files = split["val"]

assert len(train_files) == len(set(train_files)) == 216
assert len(val_files) == len(set(val_files)) == 54
assert set(train_files).isdisjoint(val_files)

if Path(split["data_path"]).resolve() != data_dir.resolve():
    raise ValueError("划分记录与当前数据目录不一致。")

current_files = {
    p.name for p in data_dir.glob("*.pth")
    if p.is_file()
}
if current_files != set(train_files + val_files):
    raise ValueError("当前特征文件名单与A0划分不一致。")

b2_save_path = (
    project_dir
    / "model/MIL/Ablation_NEW/CAMELYON16/"
      "B2_GAT3_Joint_patience10"
)
b2_save_path.mkdir(parents=True, exist_ok=True)

for filename in (
    "model.pth",
    "config.json",
    "split_on_CAMELYON16.json",
    "log.log",
    "training_summary.json",
):
    if (b2_save_path / filename).exists():
        raise FileExistsError(
            f"已有实验文件，请先检查或备份：{b2_save_path / filename}"
        )

# 保留原类及默认参数，仅指定三层配置
model_gnn = GNNModel(
    384, 256, 256, num_layers=3
)
model_iclassifier = FCLayer(256, 2)
model_bclassifier = BClassifier(256, 2)
model_mil = MILNet(model_iclassifier, model_bclassifier)
model = B2GAT3Model(model_gnn, model_mil).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4
)

print("本次图编码器：")
print(model_gnn)

config = {
    "experiment": "B2_GAT3_Joint_patience10",
    "dataset": "CAMELYON16",
    "class_num": 2,
    "feature_dim": 384,
    "reference_run": str(a0_dir),
    "split_source": str(split_source),
    "data_path": str(data_dir),
    "graph_encoder": "GNNModel(384,256,256,num_layers=3)",
    "graph_structure": str(model_gnn),
    "graph_edges": "eight-neighbor; internal GAT self-loops",
    "aggregation_head": "original BClassifier(256,2)",
    "training_loss": "0.5 bag CE + 0.5 max-instance CE",
    "optimizer": "Adam",
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "patience": 10,
    "early_stopping_logic": (
        "reset to 10 on improvement, then decrement every epoch"
    ),
    "consecutive_non_improvements_to_stop": 9,
    "checkpoint_selection": "minimum validation bag CE",
    "training_seed": None,
    "train_order": "fixed saved split order; no reshuffling",
}

with (b2_save_path / "config.json").open(
    "x", encoding="utf-8"
) as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

(b2_save_path / "split_on_CAMELYON16.json").write_bytes(
    split_source.read_bytes()
)

logger = logging.getLogger("B2_GAT3_CAMELYON16")
logger.setLevel(logging.INFO)
logger.propagate = False

for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

handler = logging.FileHandler(
    b2_save_path / "log.log",
    mode="a",
    encoding="utf-8",
)
handler.setFormatter(
    logging.Formatter("%(asctime)s - %(message)s")
)
logger.addHandler(handler)
logger.info(json.dumps(config, ensure_ascii=False))

print("实验：B2，八邻域GAT3＋原聚合器＋联合CE", flush=True)
print("划分来源：", split_source, flush=True)
print("保存目录：", b2_save_path, flush=True)

best_val_ce = float("inf")
best_epoch = None
remaining = 10

try:
    for epoch in range(100):
        model.train()
        train_losses = []

        for filename in tqdm(
            train_files, desc=f"B2 GAT3 Epoch {epoch} Train"
        ):
            features, edges, target = get_a2_data(
                data_dir / filename, device
            )
            optimizer.zero_grad()

            instance_logits, bag_logits, _, _ = model(
                features, edges
            )

            assert tuple(bag_logits.shape) == (1, 2)
            assert tuple(instance_logits.shape) == (
                features.shape[0], 2
            )

            max_logits = instance_logits.max(
                dim=0, keepdim=True
            ).values

            loss = (
                0.5 * F.cross_entropy(bag_logits, target)
                + 0.5 * F.cross_entropy(max_logits, target)
            )

            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"非有限训练损失：{filename}"
                )

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for filename in tqdm(
                val_files, desc=f"B2 GAT3 Epoch {epoch} Val"
            ):
                features, edges, target = get_a2_data(
                    data_dir / filename, device
                )
                _, bag_logits, _, _ = model(features, edges)

                loss = F.cross_entropy(bag_logits, target)
                if not torch.isfinite(loss).item():
                    raise FloatingPointError(
                        f"非有限验证损失：{filename}"
                    )

                val_losses.append(loss.item())

        train_ce = float(np.mean(train_losses))
        val_ce = float(np.mean(val_losses))
        improved = val_ce < best_val_ce

        if improved:
            best_val_ce = val_ce
            best_epoch = epoch
            remaining = 10
            torch.save(model, b2_save_path / "model.pth")

        # 沿用A0的早停计数逻辑
        remaining -= 1

        info = (
            f"epoch:{epoch}\tremaining:{remaining}\t"
            f"train:{train_ce:.6f}\tval:{val_ce:.6f}\t"
            + ("saved" if improved else "")
        )
        print(info, flush=True)
        logger.info(info)
        handler.flush()

        if remaining < 1:
            print("Early stopping", flush=True)
            logger.info("Early stopping")
            break

    summary = {
        "experiment": config["experiment"],
        "best_epoch": best_epoch,
        "best_validation_ce": best_val_ce,
        "weight_path": str(b2_save_path / "model.pth"),
    }

    with (b2_save_path / "training_summary.json").open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    logger.info("Training complete: %s", summary)
    print("\n训练完成：", summary)

finally:
    handler.flush()
    handler.close()
    logger.removeHandler(handler)

In [ ]:
# ============================================================
# B2消融实验测试：CAMELYON16，三层GAT + 原聚合器
# 输入：八邻域图，保持原patch顺序
# 预测：最佳权重，包分支argmax
# 指标：类别1（肿瘤）的Pre、Rec、F1及总体Acc
# ============================================================

from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

run_dir = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/"
    "Ablation_NEW/CAMELYON16/B2_GAT3_Joint_patience10/"
)
test_dir = Path(
    "/data/database/CAMELYON16/pkl_data/testing/20xbags/"
)

# 若重启过内核，先恢复读取函数及模型类定义，
# 不要重新执行整个训练单元。
required = ["get_a2_data", "B2GAT3Model", "GNNModel", "MILNet",
            "FCLayer", "BClassifier"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"请先恢复这些定义：{missing}")

for filename in (
    "model.pth",
    "config.json",
    "training_summary.json",
):
    if not (run_dir / filename).is_file():
        raise FileNotFoundError(run_dir / filename)

test_config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)
assert test_config["experiment"] == "B2_GAT3_Joint_patience10"

model = torch.load(
    run_dir / "model.pth",
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

test_files = sorted(
    p for p in test_dir.iterdir()
    if p.is_file() and p.suffix == ".pth"
)
assert len(test_files) == 129

labels, predictions, rows = [], [], []

with torch.no_grad():
    for path in tqdm(test_files, desc="B2 GAT3 Test"):
        features, edges, target = get_a2_data(path, device)
        _, bag_logits, _, _ = model(features, edges)

        assert tuple(bag_logits.shape) == (1, 2)
        assert torch.isfinite(bag_logits).all()

        true_label = int(target.item())
        prediction = int(bag_logits.argmax(dim=1).item())

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": path.name,
            "label": true_label,
            "prediction": prediction,
            "tumor_probability": float(
                bag_logits.softmax(dim=1)[0, 1].item()
            ),
        })

assert [labels.count(i) for i in (0, 1)] == [81, 48]

result = {
    "model": "CAMELYON16_B2_GAT3_Joint_patience10",
    "weight_path": str((run_dir / "model.pth").resolve()),
    "test_data_path": str(test_dir.resolve()),
    "n": len(labels),
    "average": "binary",
    "positive_label": 1,
    "precision": float(precision_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "accuracy": float(accuracy_score(labels, predictions)),
    "recall": float(recall_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "confusion_matrix_labels_0_1": confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别；顺序0、1")
print(np.array(result["confusion_matrix_labels_0_1"]))

# 重复运行更新测试JSON，不修改模型权重
for filename, content in (
    ("test_metrics.json", result),
    ("test_predictions.json", rows),
):
    with (run_dir / filename).open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

log_path = run_dir / "log.log"
with log_path.open("a", encoding="utf-8") as f:
    f.write("\n" + summary + "\n")
    f.write(
        "Confusion matrix [0, 1]:\n"
        + str(result["confusion_matrix_labels_0_1"])
        + "\n"
    )

assert summary in log_path.read_text(encoding="utf-8")
print("测试结果和日志已保存至：", run_dir.resolve())

In [ ]:
# ============================================================
# B3 消融实验：CAMELYON16，一层GCN
# 模型：八邻域GCN1 + 原关键实例聚合器
# 监督：0.5包级CE + 0.5最大实例CE
# 相对A0：将一层GAT替换为一层GCN，输出维度保持256
# GCN使用标准归一化及内部自环，不传GAT专用参数
# ============================================================

import json
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as geom_nn
from tqdm import tqdm

from wsi.nets.dasmil import (
    FCLayer,
    BClassifier,
    MILNet,
)

if "get_a2_data" not in globals():
    raise NameError(
        "请先执行A2的单元1，定义八邻域读取函数get_a2_data。"
    )


class B3GCNEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = geom_nn.GCNConv(
            in_channels=384,
            out_channels=256,
            improved=False,
            cached=False,
            add_self_loops=True,
            normalize=True,
            bias=True,
        )

    def forward(self, features, edges):
        return self.conv(features, edges)


class B3GCN1Model(nn.Module):
    def __init__(self, gnn, mil):
        super().__init__()
        self.gnn = gnn
        self.mil = mil

    def forward(self, features, edges):
        return self.mil(self.gnn(features, edges))


print("B3一层GCN模型已定义")

In [ ]:
# ============================================================
# B3训练：八邻域GCN1 + 原聚合器 + 联合CE
# 划分：沿用A0，216训练 / 54验证
# 优化：Adam，lr=3e-4，最多100轮，patience=10
# 选模：最低验证包级CE
# 保存：Ablation_NEW/CAMELYON16/B3_GCN1_Joint_patience10
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA不可用，请检查GPU和Notebook内核。")

device = torch.device("cuda:0")

project_dir = Path("/data/code/WSI/WSI_pretrain")
data_dir = Path(
    "/data/database/CAMELYON16/pkl_data/training/20xbags/"
)
a0_dir = (
    project_dir
    / "model/MIL/compare/Camelyon/Ours/Ours_patience10"
)

split_candidates = [
    a0_dir / "split_on_CAMELYON16.json",
    a0_dir / "split.json",
    project_dir / (
        "model/MIL/compare/Camelyon/"
        "patience10_split_on_CAMELYON16.json"
    ),
    project_dir / (
        "model/MIL/compare/Camelyon/patience10_split.json"
    ),
]

split_source = next(
    (p for p in split_candidates if p.is_file()),
    None,
)
if split_source is None:
    raise FileNotFoundError("未找到A0或公共划分，不重新随机划分。")

split = json.loads(
    split_source.read_text(encoding="utf-8")
)
train_files = split["train"]
val_files = split["val"]

assert len(train_files) == len(set(train_files)) == 216
assert len(val_files) == len(set(val_files)) == 54
assert set(train_files).isdisjoint(val_files)

if Path(split["data_path"]).resolve() != data_dir.resolve():
    raise ValueError("划分记录与当前数据目录不一致。")

current_files = {
    p.name for p in data_dir.glob("*.pth")
    if p.is_file()
}
if current_files != set(train_files + val_files):
    raise ValueError("当前特征文件名单与A0划分不一致。")

b3_save_path = (
    project_dir
    / "model/MIL/Ablation_NEW/CAMELYON16/"
      "B3_GCN1_Joint_patience10"
)
b3_save_path.mkdir(parents=True, exist_ok=True)

for filename in (
    "model.pth",
    "config.json",
    "split_on_CAMELYON16.json",
    "log.log",
    "training_summary.json",
):
    if (b3_save_path / filename).exists():
        raise FileExistsError(
            f"已有实验文件，请先检查或备份：{b3_save_path / filename}"
        )

model_gnn = B3GCNEncoder()
model_iclassifier = FCLayer(256, 2)
model_bclassifier = BClassifier(256, 2)
model_mil = MILNet(model_iclassifier, model_bclassifier)
model = B3GCN1Model(model_gnn, model_mil).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4
)

config = {
    "experiment": "B3_GCN1_Joint_patience10",
    "dataset": "CAMELYON16",
    "class_num": 2,
    "feature_dim": 384,
    "reference_run": str(a0_dir),
    "split_source": str(split_source),
    "data_path": str(data_dir),
    "graph_encoder": "GCNConv(384,256), one layer",
    "graph_structure": str(model_gnn),
    "graph_edges": "eight-neighbor; internal GCN self-loops",
    "gcn_normalize": True,
    "gcn_cached": False,
    "aggregation_head": "original BClassifier(256,2)",
    "training_loss": "0.5 bag CE + 0.5 max-instance CE",
    "optimizer": "Adam",
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "patience": 10,
    "early_stopping_logic": (
        "reset to 10 on improvement, then decrement every epoch"
    ),
    "consecutive_non_improvements_to_stop": 9,
    "checkpoint_selection": "minimum validation bag CE",
    "training_seed": None,
    "train_order": "fixed saved split order; no reshuffling",
}

with (b3_save_path / "config.json").open(
    "x", encoding="utf-8"
) as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

(b3_save_path / "split_on_CAMELYON16.json").write_bytes(
    split_source.read_bytes()
)

logger = logging.getLogger("B3_GCN1_CAMELYON16")
logger.setLevel(logging.INFO)
logger.propagate = False

for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

handler = logging.FileHandler(
    b3_save_path / "log.log",
    mode="a",
    encoding="utf-8",
)
handler.setFormatter(
    logging.Formatter("%(asctime)s - %(message)s")
)
logger.addHandler(handler)
logger.info(json.dumps(config, ensure_ascii=False))

print("实验：B3，八邻域GCN1＋原聚合器＋联合CE", flush=True)
print("图编码器：", model_gnn, flush=True)
print("划分来源：", split_source, flush=True)
print("保存目录：", b3_save_path, flush=True)

best_val_ce = float("inf")
best_epoch = None
remaining = 10

try:
    for epoch in range(100):
        model.train()
        train_losses = []

        for filename in tqdm(
            train_files, desc=f"B3 GCN1 Epoch {epoch} Train"
        ):
            features, edges, target = get_a2_data(
                data_dir / filename, device
            )
            optimizer.zero_grad()

            instance_logits, bag_logits, _, _ = model(
                features, edges
            )

            assert tuple(bag_logits.shape) == (1, 2)
            assert tuple(instance_logits.shape) == (
                features.shape[0], 2
            )

            max_logits = instance_logits.max(
                dim=0, keepdim=True
            ).values

            loss = (
                0.5 * F.cross_entropy(bag_logits, target)
                + 0.5 * F.cross_entropy(max_logits, target)
            )

            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"非有限训练损失：{filename}"
                )

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for filename in tqdm(
                val_files, desc=f"B3 GCN1 Epoch {epoch} Val"
            ):
                features, edges, target = get_a2_data(
                    data_dir / filename, device
                )
                _, bag_logits, _, _ = model(features, edges)

                loss = F.cross_entropy(bag_logits, target)
                if not torch.isfinite(loss).item():
                    raise FloatingPointError(
                        f"非有限验证损失：{filename}"
                    )

                val_losses.append(loss.item())

        train_ce = float(np.mean(train_losses))
        val_ce = float(np.mean(val_losses))
        improved = val_ce < best_val_ce

        if improved:
            best_val_ce = val_ce
            best_epoch = epoch
            remaining = 10
            torch.save(model, b3_save_path / "model.pth")

        remaining -= 1

        info = (
            f"epoch:{epoch}\tremaining:{remaining}\t"
            f"train:{train_ce:.6f}\tval:{val_ce:.6f}\t"
            + ("saved" if improved else "")
        )
        print(info, flush=True)
        logger.info(info)
        handler.flush()

        if remaining < 1:
            print("Early stopping", flush=True)
            logger.info("Early stopping")
            break

    summary = {
        "experiment": config["experiment"],
        "best_epoch": best_epoch,
        "best_validation_ce": best_val_ce,
        "weight_path": str(b3_save_path / "model.pth"),
    }

    with (b3_save_path / "training_summary.json").open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    logger.info("Training complete: %s", summary)
    print("\n训练完成：", summary)

finally:
    handler.flush()
    handler.close()
    logger.removeHandler(handler)

In [ ]:
# ============================================================
# B3消融实验测试：CAMELYON16，一层GCN + 原聚合器
# 输入：八邻域图，内部保留自环
# 预测：最佳权重，包分支argmax
# 指标：肿瘤类别1的Pre、Rec、F1及总体Acc
# ============================================================

from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

run_dir = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/"
    "Ablation_NEW/CAMELYON16/B3_GCN1_Joint_patience10/"
)
test_dir = Path(
    "/data/database/CAMELYON16/pkl_data/testing/20xbags/"
)

# 重启内核后，先恢复A2读取函数及B3单元1的模型定义
required = [
    "get_a2_data",
    "B3GCNEncoder",
    "B3GCN1Model",
    "FCLayer",
    "BClassifier",
    "MILNet",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"请先恢复这些定义：{missing}")

for filename in (
    "model.pth",
    "config.json",
    "training_summary.json",
):
    if not (run_dir / filename).is_file():
        raise FileNotFoundError(run_dir / filename)

test_config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)
assert test_config["experiment"] == "B3_GCN1_Joint_patience10"

model = torch.load(
    run_dir / "model.pth",
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

test_files = sorted(
    p for p in test_dir.iterdir()
    if p.is_file() and p.suffix == ".pth"
)
assert len(test_files) == 129

labels, predictions, rows = [], [], []

with torch.no_grad():
    for path in tqdm(test_files, desc="B3 GCN1 Test"):
        features, edges, target = get_a2_data(path, device)
        _, bag_logits, _, _ = model(features, edges)

        assert tuple(bag_logits.shape) == (1, 2)
        assert torch.isfinite(bag_logits).all()

        true_label = int(target.item())
        prediction = int(bag_logits.argmax(dim=1).item())

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": path.name,
            "label": true_label,
            "prediction": prediction,
            "tumor_probability": float(
                bag_logits.softmax(dim=1)[0, 1].item()
            ),
        })

assert [labels.count(i) for i in (0, 1)] == [81, 48]

result = {
    "model": "CAMELYON16_B3_GCN1_Joint_patience10",
    "weight_path": str((run_dir / "model.pth").resolve()),
    "test_data_path": str(test_dir.resolve()),
    "n": len(labels),
    "average": "binary",
    "positive_label": 1,
    "precision": float(precision_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "accuracy": float(accuracy_score(labels, predictions)),
    "recall": float(recall_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "confusion_matrix_labels_0_1": confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别；顺序0、1")
print(np.array(result["confusion_matrix_labels_0_1"]))

for filename, content in (
    ("test_metrics.json", result),
    ("test_predictions.json", rows),
):
    with (run_dir / filename).open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

log_path = run_dir / "log.log"
with log_path.open("a", encoding="utf-8") as f:
    f.write("\n" + summary + "\n")
    f.write(
        "Confusion matrix [0, 1]:\n"
        + str(result["confusion_matrix_labels_0_1"])
        + "\n"
    )

assert summary in log_path.read_text(encoding="utf-8")
print("测试结果和日志已保存至：", run_dir.resolve())

In [ ]:
# ============================================================
# B4 消融实验：CAMELYON16，两层GCN
# 模型：GCN(384→256) + ReLU + Dropout(0.1)
#       + GCN(256→256) + 原关键实例聚合器
# 空间连接：八邻域，GCN内部保留自环
# 监督：0.5包级CE + 0.5最大实例CE
# ============================================================

import json
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as geom_nn
from tqdm import tqdm

from wsi.nets.dasmil import (
    FCLayer,
    BClassifier,
    MILNet,
)

if "get_a2_data" not in globals():
    raise NameError(
        "请先执行A2的单元1，定义八邻域读取函数get_a2_data。"
    )


class B4GCNEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = geom_nn.GCNConv(
            384, 256,
            improved=False,
            cached=False,
            add_self_loops=True,
            normalize=True,
            bias=True,
        )
        self.activation = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.1)
        self.conv2 = geom_nn.GCNConv(
            256, 256,
            improved=False,
            cached=False,
            add_self_loops=True,
            normalize=True,
            bias=True,
        )

    def forward(self, features, edges):
        features = self.conv1(features, edges)
        features = self.activation(features)
        features = self.dropout(features)
        return self.conv2(features, edges)


class B4GCN2Model(nn.Module):
    def __init__(self, gnn, mil):
        super().__init__()
        self.gnn = gnn
        self.mil = mil

    def forward(self, features, edges):
        return self.mil(self.gnn(features, edges))


print("B4两层GCN模型已定义")

In [ ]:
# ============================================================
# B4训练：八邻域GCN2 + 原聚合器 + 联合CE
# 划分：沿用A0，216训练 / 54验证，不重新划分
# 优化：Adam，lr=3e-4，最多100轮，patience=10
# 选模：最低验证包级CE
# 保存：Ablation_NEW/CAMELYON16/B4_GCN2_Joint_patience10
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA不可用，请检查GPU和Notebook内核。")

device = torch.device("cuda:0")

project_dir = Path("/data/code/WSI/WSI_pretrain")
data_dir = Path(
    "/data/database/CAMELYON16/pkl_data/training/20xbags/"
)
a0_dir = (
    project_dir
    / "model/MIL/compare/Camelyon/Ours/Ours_patience10"
)

split_candidates = [
    a0_dir / "split_on_CAMELYON16.json",
    a0_dir / "split.json",
    project_dir / (
        "model/MIL/compare/Camelyon/"
        "patience10_split_on_CAMELYON16.json"
    ),
    project_dir / (
        "model/MIL/compare/Camelyon/patience10_split.json"
    ),
]

split_source = next(
    (p for p in split_candidates if p.is_file()),
    None,
)
if split_source is None:
    raise FileNotFoundError("未找到A0或公共划分，不重新随机划分。")

split = json.loads(
    split_source.read_text(encoding="utf-8")
)
train_files = split["train"]
val_files = split["val"]

assert len(train_files) == len(set(train_files)) == 216
assert len(val_files) == len(set(val_files)) == 54
assert set(train_files).isdisjoint(val_files)

if Path(split["data_path"]).resolve() != data_dir.resolve():
    raise ValueError("划分记录与当前数据目录不一致。")

current_files = {
    p.name for p in data_dir.glob("*.pth")
    if p.is_file()
}
if current_files != set(train_files + val_files):
    raise ValueError("当前特征文件名单与A0划分不一致。")

b4_save_path = (
    project_dir
    / "model/MIL/Ablation_NEW/CAMELYON16/"
      "B4_GCN2_Joint_patience10"
)
b4_save_path.mkdir(parents=True, exist_ok=True)

for filename in (
    "model.pth",
    "config.json",
    "split_on_CAMELYON16.json",
    "log.log",
    "training_summary.json",
):
    if (b4_save_path / filename).exists():
        raise FileExistsError(
            f"已有实验文件，请先检查或备份：{b4_save_path / filename}"
        )

model_gnn = B4GCNEncoder()
model_iclassifier = FCLayer(256, 2)
model_bclassifier = BClassifier(256, 2)
model_mil = MILNet(model_iclassifier, model_bclassifier)
model = B4GCN2Model(model_gnn, model_mil).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4
)

config = {
    "experiment": "B4_GCN2_Joint_patience10",
    "dataset": "CAMELYON16",
    "class_num": 2,
    "feature_dim": 384,
    "reference_run": str(a0_dir),
    "split_source": str(split_source),
    "data_path": str(data_dir),
    "graph_encoder": (
        "GCNConv(384,256)-ReLU-Dropout(0.1)-GCNConv(256,256)"
    ),
    "graph_structure": str(model_gnn),
    "graph_edges": "eight-neighbor; internal GCN self-loops",
    "gcn_normalize": True,
    "gcn_cached": False,
    "aggregation_head": "original BClassifier(256,2)",
    "training_loss": "0.5 bag CE + 0.5 max-instance CE",
    "optimizer": "Adam",
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "patience": 10,
    "early_stopping_logic": (
        "reset to 10 on improvement, then decrement every epoch"
    ),
    "consecutive_non_improvements_to_stop": 9,
    "checkpoint_selection": "minimum validation bag CE",
    "training_seed": None,
    "train_order": "fixed saved split order; no reshuffling",
}

with (b4_save_path / "config.json").open(
    "x", encoding="utf-8"
) as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

(b4_save_path / "split_on_CAMELYON16.json").write_bytes(
    split_source.read_bytes()
)

logger = logging.getLogger("B4_GCN2_CAMELYON16")
logger.setLevel(logging.INFO)
logger.propagate = False

for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

handler = logging.FileHandler(
    b4_save_path / "log.log",
    mode="a",
    encoding="utf-8",
)
handler.setFormatter(
    logging.Formatter("%(asctime)s - %(message)s")
)
logger.addHandler(handler)
logger.info(json.dumps(config, ensure_ascii=False))

print("实验：B4，八邻域GCN2＋原聚合器＋联合CE", flush=True)
print("图编码器：", model_gnn, flush=True)
print("划分来源：", split_source, flush=True)
print("保存目录：", b4_save_path, flush=True)

best_val_ce = float("inf")
best_epoch = None
remaining = 10

try:
    for epoch in range(100):
        model.train()
        train_losses = []

        for filename in tqdm(
            train_files, desc=f"B4 GCN2 Epoch {epoch} Train"
        ):
            features, edges, target = get_a2_data(
                data_dir / filename, device
            )
            optimizer.zero_grad()

            instance_logits, bag_logits, _, _ = model(
                features, edges
            )

            assert tuple(bag_logits.shape) == (1, 2)
            assert tuple(instance_logits.shape) == (
                features.shape[0], 2
            )

            max_logits = instance_logits.max(
                dim=0, keepdim=True
            ).values

            loss = (
                0.5 * F.cross_entropy(bag_logits, target)
                + 0.5 * F.cross_entropy(max_logits, target)
            )

            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"非有限训练损失：{filename}"
                )

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for filename in tqdm(
                val_files, desc=f"B4 GCN2 Epoch {epoch} Val"
            ):
                features, edges, target = get_a2_data(
                    data_dir / filename, device
                )
                _, bag_logits, _, _ = model(features, edges)

                loss = F.cross_entropy(bag_logits, target)
                if not torch.isfinite(loss).item():
                    raise FloatingPointError(
                        f"非有限验证损失：{filename}"
                    )

                val_losses.append(loss.item())

        train_ce = float(np.mean(train_losses))
        val_ce = float(np.mean(val_losses))
        improved = val_ce < best_val_ce

        if improved:
            best_val_ce = val_ce
            best_epoch = epoch
            remaining = 10
            torch.save(model, b4_save_path / "model.pth")

        # 沿用A0的计数逻辑：改善后重置，再逐轮减1
        remaining -= 1

        info = (
            f"epoch:{epoch}\tremaining:{remaining}\t"
            f"train:{train_ce:.6f}\tval:{val_ce:.6f}\t"
            + ("saved" if improved else "")
        )
        print(info, flush=True)
        logger.info(info)
        handler.flush()

        if remaining < 1:
            print("Early stopping", flush=True)
            logger.info("Early stopping")
            break

    summary = {
        "experiment": config["experiment"],
        "best_epoch": best_epoch,
        "best_validation_ce": best_val_ce,
        "weight_path": str(b4_save_path / "model.pth"),
    }

    with (b4_save_path / "training_summary.json").open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    logger.info("Training complete: %s", summary)
    print("\n训练完成：", summary)

finally:
    handler.flush()
    handler.close()
    logger.removeHandler(handler)

In [ ]:
# ============================================================
# B4消融实验测试：CAMELYON16，两层GCN + 原聚合器
# 输入：八邻域图，GCN内部保留自环
# 预测：最佳权重，包分支argmax
# 指标：肿瘤类别1的Pre、Rec、F1及总体Acc
# ============================================================

from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

run_dir = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/"
    "Ablation_NEW/CAMELYON16/B4_GCN2_Joint_patience10/"
)
test_dir = Path(
    "/data/database/CAMELYON16/pkl_data/testing/20xbags/"
)

# 重启后先恢复A2读取函数及本实验单元1
required = [
    "get_a2_data",
    "B4GCNEncoder",
    "B4GCN2Model",
    "FCLayer",
    "BClassifier",
    "MILNet",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"请先恢复这些定义：{missing}")

for filename in (
    "model.pth",
    "config.json",
    "training_summary.json",
):
    if not (run_dir / filename).is_file():
        raise FileNotFoundError(run_dir / filename)

test_config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)
assert test_config["experiment"] == "B4_GCN2_Joint_patience10"

model = torch.load(
    run_dir / "model.pth",
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

test_files = sorted(
    p for p in test_dir.iterdir()
    if p.is_file() and p.suffix == ".pth"
)
assert len(test_files) == 129

labels, predictions, rows = [], [], []

with torch.no_grad():
    for path in tqdm(test_files, desc="B4 GCN2 Test"):
        features, edges, target = get_a2_data(path, device)
        _, bag_logits, _, _ = model(features, edges)

        assert tuple(bag_logits.shape) == (1, 2)
        assert torch.isfinite(bag_logits).all()

        true_label = int(target.item())
        prediction = int(bag_logits.argmax(dim=1).item())

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": path.name,
            "label": true_label,
            "prediction": prediction,
            "tumor_probability": float(
                bag_logits.softmax(dim=1)[0, 1].item()
            ),
        })

assert [labels.count(i) for i in (0, 1)] == [81, 48]

result = {
    "model": "CAMELYON16_B4_GCN2_Joint_patience10",
    "weight_path": str((run_dir / "model.pth").resolve()),
    "test_data_path": str(test_dir.resolve()),
    "n": len(labels),
    "average": "binary",
    "positive_label": 1,
    "precision": float(precision_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "accuracy": float(accuracy_score(labels, predictions)),
    "recall": float(recall_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "confusion_matrix_labels_0_1": confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别；顺序0、1")
print(np.array(result["confusion_matrix_labels_0_1"]))

# 重复运行更新测试JSON，不修改模型权重
for filename, content in (
    ("test_metrics.json", result),
    ("test_predictions.json", rows),
):
    with (run_dir / filename).open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

log_path = run_dir / "log.log"
with log_path.open("a", encoding="utf-8") as f:
    f.write("\n" + summary + "\n")
    f.write(
        "Confusion matrix [0, 1]:\n"
        + str(result["confusion_matrix_labels_0_1"])
        + "\n"
    )

assert summary in log_path.read_text(encoding="utf-8")
print("测试结果和日志已保存至：", run_dir.resolve())

In [ ]:
# ============================================================
# B5 消融实验：CAMELYON16，三层GCN
# 模型：GCN(384→256) + ReLU + Dropout(0.1)
#       + GCN(256→256) + ReLU + Dropout(0.1)
#       + GCN(256→256) + 原关键实例聚合器
# 空间连接：八邻域，各GCN层内部保留自环
# 监督：0.5包级CE + 0.5最大实例CE
# ============================================================

import json
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as geom_nn
from tqdm import tqdm

from wsi.nets.dasmil import (
    FCLayer,
    BClassifier,
    MILNet,
)

if "get_a2_data" not in globals():
    raise NameError(
        "请先执行A2的单元1，定义八邻域读取函数get_a2_data。"
    )


class B5GCNEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = geom_nn.GCNConv(
            384, 256,
            improved=False,
            cached=False,
            add_self_loops=True,
            normalize=True,
            bias=True,
        )
        self.activation1 = nn.ReLU(inplace=True)
        self.dropout1 = nn.Dropout(0.1)

        self.conv2 = geom_nn.GCNConv(
            256, 256,
            improved=False,
            cached=False,
            add_self_loops=True,
            normalize=True,
            bias=True,
        )
        self.activation2 = nn.ReLU(inplace=True)
        self.dropout2 = nn.Dropout(0.1)

        self.conv3 = geom_nn.GCNConv(
            256, 256,
            improved=False,
            cached=False,
            add_self_loops=True,
            normalize=True,
            bias=True,
        )

    def forward(self, features, edges):
        features = self.conv1(features, edges)
        features = self.activation1(features)
        features = self.dropout1(features)

        features = self.conv2(features, edges)
        features = self.activation2(features)
        features = self.dropout2(features)

        return self.conv3(features, edges)


class B5GCN3Model(nn.Module):
    def __init__(self, gnn, mil):
        super().__init__()
        self.gnn = gnn
        self.mil = mil

    def forward(self, features, edges):
        return self.mil(self.gnn(features, edges))


print("B5三层GCN模型已定义")

In [ ]:
# ============================================================
# B5训练：八邻域GCN3 + 原聚合器 + 联合CE
# 划分：沿用A0，216训练 / 54验证，不重新划分
# 优化：Adam，lr=3e-4，最多100轮，patience=10
# 选模：最低验证包级CE
# 保存：Ablation_NEW/CAMELYON16/B5_GCN3_Joint_patience10
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA不可用，请检查GPU和Notebook内核。")

device = torch.device("cuda:0")

project_dir = Path("/data/code/WSI/WSI_pretrain")
data_dir = Path(
    "/data/database/CAMELYON16/pkl_data/training/20xbags/"
)
a0_dir = (
    project_dir
    / "model/MIL/compare/Camelyon/Ours/Ours_patience10"
)

split_candidates = [
    a0_dir / "split_on_CAMELYON16.json",
    a0_dir / "split.json",
    project_dir / (
        "model/MIL/compare/Camelyon/"
        "patience10_split_on_CAMELYON16.json"
    ),
    project_dir / (
        "model/MIL/compare/Camelyon/patience10_split.json"
    ),
]

split_source = next(
    (p for p in split_candidates if p.is_file()),
    None,
)
if split_source is None:
    raise FileNotFoundError("未找到A0或公共划分，不重新随机划分。")

split = json.loads(
    split_source.read_text(encoding="utf-8")
)
train_files = split["train"]
val_files = split["val"]

assert len(train_files) == len(set(train_files)) == 216
assert len(val_files) == len(set(val_files)) == 54
assert set(train_files).isdisjoint(val_files)

if Path(split["data_path"]).resolve() != data_dir.resolve():
    raise ValueError("划分记录与当前数据目录不一致。")

current_files = {
    p.name for p in data_dir.glob("*.pth")
    if p.is_file()
}
if current_files != set(train_files + val_files):
    raise ValueError("当前特征文件名单与A0划分不一致。")

b5_save_path = (
    project_dir
    / "model/MIL/Ablation_NEW/CAMELYON16/"
      "B5_GCN3_Joint_patience10"
)
b5_save_path.mkdir(parents=True, exist_ok=True)

for filename in (
    "model.pth",
    "config.json",
    "split_on_CAMELYON16.json",
    "log.log",
    "training_summary.json",
):
    if (b5_save_path / filename).exists():
        raise FileExistsError(
            f"已有实验文件，请先检查或备份：{b5_save_path / filename}"
        )

model_gnn = B5GCNEncoder()
model_iclassifier = FCLayer(256, 2)
model_bclassifier = BClassifier(256, 2)
model_mil = MILNet(model_iclassifier, model_bclassifier)
model = B5GCN3Model(model_gnn, model_mil).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4
)

config = {
    "experiment": "B5_GCN3_Joint_patience10",
    "dataset": "CAMELYON16",
    "class_num": 2,
    "feature_dim": 384,
    "reference_run": str(a0_dir),
    "split_source": str(split_source),
    "data_path": str(data_dir),
    "graph_encoder": (
        "GCNConv(384,256)-ReLU-Dropout(0.1)-"
        "GCNConv(256,256)-ReLU-Dropout(0.1)-"
        "GCNConv(256,256)"
    ),
    "graph_structure": str(model_gnn),
    "graph_edges": "eight-neighbor; internal GCN self-loops",
    "gcn_normalize": True,
    "gcn_cached": False,
    "aggregation_head": "original BClassifier(256,2)",
    "training_loss": "0.5 bag CE + 0.5 max-instance CE",
    "optimizer": "Adam",
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "patience": 10,
    "early_stopping_logic": (
        "reset to 10 on improvement, then decrement every epoch"
    ),
    "consecutive_non_improvements_to_stop": 9,
    "checkpoint_selection": "minimum validation bag CE",
    "training_seed": None,
    "train_order": "fixed saved split order; no reshuffling",
}

with (b5_save_path / "config.json").open(
    "x", encoding="utf-8"
) as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

(b5_save_path / "split_on_CAMELYON16.json").write_bytes(
    split_source.read_bytes()
)

logger = logging.getLogger("B5_GCN3_CAMELYON16")
logger.setLevel(logging.INFO)
logger.propagate = False

for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

handler = logging.FileHandler(
    b5_save_path / "log.log",
    mode="a",
    encoding="utf-8",
)
handler.setFormatter(
    logging.Formatter("%(asctime)s - %(message)s")
)
logger.addHandler(handler)
logger.info(json.dumps(config, ensure_ascii=False))

print("实验：B5，八邻域GCN3＋原聚合器＋联合CE", flush=True)
print("图编码器：", model_gnn, flush=True)
print("划分来源：", split_source, flush=True)
print("保存目录：", b5_save_path, flush=True)

best_val_ce = float("inf")
best_epoch = None
remaining = 10

try:
    for epoch in range(100):
        model.train()
        train_losses = []

        for filename in tqdm(
            train_files, desc=f"B5 GCN3 Epoch {epoch} Train"
        ):
            features, edges, target = get_a2_data(
                data_dir / filename, device
            )
            optimizer.zero_grad()

            instance_logits, bag_logits, _, _ = model(
                features, edges
            )

            assert tuple(bag_logits.shape) == (1, 2)
            assert tuple(instance_logits.shape) == (
                features.shape[0], 2
            )

            max_logits = instance_logits.max(
                dim=0, keepdim=True
            ).values

            loss = (
                0.5 * F.cross_entropy(bag_logits, target)
                + 0.5 * F.cross_entropy(max_logits, target)
            )

            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"非有限训练损失：{filename}"
                )

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for filename in tqdm(
                val_files, desc=f"B5 GCN3 Epoch {epoch} Val"
            ):
                features, edges, target = get_a2_data(
                    data_dir / filename, device
                )
                _, bag_logits, _, _ = model(features, edges)

                loss = F.cross_entropy(bag_logits, target)
                if not torch.isfinite(loss).item():
                    raise FloatingPointError(
                        f"非有限验证损失：{filename}"
                    )

                val_losses.append(loss.item())

        train_ce = float(np.mean(train_losses))
        val_ce = float(np.mean(val_losses))
        improved = val_ce < best_val_ce

        if improved:
            best_val_ce = val_ce
            best_epoch = epoch
            remaining = 10
            torch.save(model, b5_save_path / "model.pth")

        # 与A0相同的早停计数逻辑
        remaining -= 1

        info = (
            f"epoch:{epoch}\tremaining:{remaining}\t"
            f"train:{train_ce:.6f}\tval:{val_ce:.6f}\t"
            + ("saved" if improved else "")
        )
        print(info, flush=True)
        logger.info(info)
        handler.flush()

        if remaining < 1:
            print("Early stopping", flush=True)
            logger.info("Early stopping")
            break

    summary = {
        "experiment": config["experiment"],
        "best_epoch": best_epoch,
        "best_validation_ce": best_val_ce,
        "weight_path": str(b5_save_path / "model.pth"),
    }

    with (b5_save_path / "training_summary.json").open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    logger.info("Training complete: %s", summary)
    print("\n训练完成：", summary)

finally:
    handler.flush()
    handler.close()
    logger.removeHandler(handler)

In [ ]:
# ============================================================
# B5消融实验测试：CAMELYON16，三层GCN + 原聚合器
# 输入：八邻域图，GCN内部保留自环
# 预测：最佳权重，包分支argmax
# 指标：肿瘤类别1的Pre、Rec、F1及总体Acc
# ============================================================

from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

run_dir = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/"
    "Ablation_NEW/CAMELYON16/B5_GCN3_Joint_patience10/"
)
test_dir = Path(
    "/data/database/CAMELYON16/pkl_data/testing/20xbags/"
)

# 重启内核后，先恢复A2读取函数和本实验单元1
required = [
    "get_a2_data",
    "B5GCNEncoder",
    "B5GCN3Model",
    "FCLayer",
    "BClassifier",
    "MILNet",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"请先恢复这些定义：{missing}")

for filename in (
    "model.pth",
    "config.json",
    "training_summary.json",
):
    if not (run_dir / filename).is_file():
        raise FileNotFoundError(run_dir / filename)

test_config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)
assert test_config["experiment"] == "B5_GCN3_Joint_patience10"

model = torch.load(
    run_dir / "model.pth",
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

test_files = sorted(
    p for p in test_dir.iterdir()
    if p.is_file() and p.suffix == ".pth"
)
assert len(test_files) == 129

labels, predictions, rows = [], [], []

with torch.no_grad():
    for path in tqdm(test_files, desc="B5 GCN3 Test"):
        features, edges, target = get_a2_data(path, device)
        _, bag_logits, _, _ = model(features, edges)

        assert tuple(bag_logits.shape) == (1, 2)
        assert torch.isfinite(bag_logits).all()

        true_label = int(target.item())
        prediction = int(bag_logits.argmax(dim=1).item())

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": path.name,
            "label": true_label,
            "prediction": prediction,
            "tumor_probability": float(
                bag_logits.softmax(dim=1)[0, 1].item()
            ),
        })

assert [labels.count(i) for i in (0, 1)] == [81, 48]

result = {
    "model": "CAMELYON16_B5_GCN3_Joint_patience10",
    "weight_path": str((run_dir / "model.pth").resolve()),
    "test_data_path": str(test_dir.resolve()),
    "n": len(labels),
    "average": "binary",
    "positive_label": 1,
    "precision": float(precision_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "accuracy": float(accuracy_score(labels, predictions)),
    "recall": float(recall_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "confusion_matrix_labels_0_1": confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别；顺序0、1")
print(np.array(result["confusion_matrix_labels_0_1"]))

for filename, content in (
    ("test_metrics.json", result),
    ("test_predictions.json", rows),
):
    with (run_dir / filename).open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

log_path = run_dir / "log.log"
with log_path.open("a", encoding="utf-8") as f:
    f.write("\n" + summary + "\n")
    f.write(
        "Confusion matrix [0, 1]:\n"
        + str(result["confusion_matrix_labels_0_1"])
        + "\n"
    )

assert summary in log_path.read_text(encoding="utf-8")
print("测试结果和日志已保存至：", run_dir.resolve())

In [ ]:
# ============================================================
# C1尺度配置实验：10尺度特征准备与划分映射
# 目标：沿用A0相同切片的216训练 / 54验证 / 129测试
# 文件对应：切片ID，不使用包含patch数量的完整文件名匹配
# 本单元只核查、准备映射，不训练
# ============================================================

import json
from pathlib import Path

import torch

project_dir = Path("/data/code/WSI/WSI_pretrain")
base_dir = Path("/data/database/CAMELYON16/pkl_data")

a0_dir = (
    project_dir
    / "model/MIL/compare/Camelyon/Ours/Ours_patience10"
)

split_candidates = [
    a0_dir / "split_on_CAMELYON16.json",
    a0_dir / "split.json",
    project_dir / (
        "model/MIL/compare/Camelyon/"
        "patience10_split_on_CAMELYON16.json"
    ),
    project_dir / (
        "model/MIL/compare/Camelyon/patience10_split.json"
    ),
]

split_source = next(
    (p for p in split_candidates if p.is_file()),
    None,
)
if split_source is None:
    raise FileNotFoundError("没有找到A0划分，不能重新随机划分。")

a0_split = json.loads(
    split_source.read_text(encoding="utf-8")
)


def find_scale_dir(subset, scale):
    candidates = [
        base_dir / subset / f"{scale}xbags",
        base_dir / subset / str(scale),
    ]
    found = [p for p in candidates if p.is_dir()]

    if len(found) != 1:
        raise RuntimeError(
            f"{subset}的{scale}尺度应唯一对应一个目录，"
            f"当前找到：{found}"
        )
    return found[0]


c1_train_dir = find_scale_dir("training", 10)
c1_test_dir = find_scale_dir("testing", 10)


def parse_name(filename):
    slide_id, patch_count, label = Path(filename).stem.rsplit("_", 2)
    return slide_id, int(patch_count), int(label)


def make_index(folder):
    result = {}
    for path in sorted(folder.glob("*.pth")):
        slide_id, _, _ = parse_name(path.name)
        if slide_id in result:
            raise ValueError(f"重复切片ID：{folder}/{slide_id}")
        result[slide_id] = path.name
    return result


train_index = make_index(c1_train_dir)
test_index = make_index(c1_test_dir)


def map_names(original_names, target_index):
    mapped = []

    for original_name in original_names:
        slide_id, _, original_label = parse_name(original_name)

        if slide_id not in target_index:
            raise FileNotFoundError(f"目标尺度缺少切片：{slide_id}")

        target_name = target_index[slide_id]
        _, _, target_label = parse_name(target_name)

        if original_label != target_label:
            raise ValueError(f"跨尺度标签不一致：{slide_id}")

        mapped.append(target_name)

    return mapped


c1_train_files = map_names(a0_split["train"], train_index)
c1_val_files = map_names(a0_split["val"], train_index)

reference_test_dir = base_dir / "testing/20xbags"
if not reference_test_dir.is_dir():
    raise FileNotFoundError(reference_test_dir)

reference_test_files = sorted(
    p.name for p in reference_test_dir.glob("*.pth")
    if p.is_file()
)
c1_test_files = map_names(reference_test_files, test_index)

assert len(c1_train_files) == 216
assert len(c1_val_files) == 54
assert len(c1_test_files) == 129
assert set(c1_train_files).isdisjoint(c1_val_files)

assert set(c1_train_files + c1_val_files) == set(train_index.values()), (
    "10尺度开发集存在额外或不匹配的切片"
)
assert set(c1_test_files) == set(test_index.values()), (
    "10尺度测试集存在额外或不匹配的切片"
)

print("A0划分：", split_source)
print("C1训练特征：", c1_train_dir)
print("C1测试特征：", c1_test_dir)
print(
    "映射后数量：",
    len(c1_train_files),
    len(c1_val_files),
    len(c1_test_files),
)

for folder, filename in [
    (c1_train_dir, c1_train_files[0]),
    (c1_test_dir, c1_test_files[0]),
]:
    data = torch.load(
        folder / filename,
        map_location="cpu",
        weights_only=False,
    )
    print({
        "file": filename,
        "feature_shape": tuple(data["features"].shape),
        "level": data.get("level"),
        "level_base": data.get("level_base"),
        "patch_size": data.get("patch_size"),
        "label": int(torch.as_tensor(data["label"]).item()),
    })

print("\nC1划分映射完成，未重新随机划分。")

In [ ]:
# ============================================================
# 公共数据读取：保留全部patch，按该尺度坐标构建八邻域图
# 用于C1/C2等空间模型；自环由GAT内部添加
# ============================================================

import torch


def get_a2_data(path, device):
    data = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    features = data["features"].to(device)
    label = torch.as_tensor(
        data["label"],
        dtype=torch.long,
        device=device,
    ).reshape(-1)

    coords = [
        tuple(map(int, xy))
        for xy in data["rects"]
    ]
    coordinate_index = {
        xy: i for i, xy in enumerate(coords)
    }

    assert features.ndim == 2
    assert features.shape[0] > 0
    assert features.shape[1] == 384
    assert features.shape[0] == len(coords)
    assert len(coordinate_index) == len(coords), "存在重复坐标"
    assert torch.isfinite(features).all()
    assert label.numel() == 1
    assert int(label.item()) in (0, 1)

    stride = int(data["patch_size"])
    assert stride > 0

    offsets = [
        (stride, -stride),
        (stride, 0),
        (stride, stride),
        (0, -stride),
        (0, stride),
        (-stride, -stride),
        (-stride, 0),
        (-stride, stride),
    ]

    edges = []
    for i, (x, y) in enumerate(coords):
        for dx, dy in offsets:
            j = coordinate_index.get((x + dx, y + dy))
            if j is not None:
                edges.append([i, j])

    if edges:
        edge_index = torch.tensor(
            edges,
            dtype=torch.long,
            device=device,
        ).t().contiguous()
    else:
        edge_index = torch.empty(
            (2, 0),
            dtype=torch.long,
            device=device,
        )

    return features, edge_index, label


print("get_a2_data 已定义，可以重新运行C1训练单元。")

In [ ]:
# ============================================================
# C1 尺度配置实验：CAMELYON16，10xbags / Level 2
# 模型：八邻域GAT1 + 原关键实例聚合器
# 监督：0.5包级CE + 0.5最大实例CE
# 划分：按切片ID沿用A0，216训练 / 54验证
# 优化：Adam，lr=3e-4，最多100轮，patience=10
# 选模：最低验证包级CE
# 保存：Ablation_NEW/CAMELYON16/C1_Level2_Joint_patience10
# ============================================================

import json
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

from wsi.nets.dasmil import (
    GNNModel,
    FCLayer,
    BClassifier,
    MILNet,
)

# 先运行前面的C1划分映射单元及A2数据读取定义
required = [
    "get_a2_data",
    "c1_train_dir",
    "c1_test_dir",
    "c1_train_files",
    "c1_val_files",
    "c1_test_files",
    "a0_split",
    "split_source",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"请先恢复C1划分映射及读取函数：{missing}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA不可用，请检查GPU和Notebook内核。")

device = torch.device("cuda:0")

train_dir_c1 = Path(c1_train_dir)
test_dir_c1 = Path(c1_test_dir)

# 保存当前名单副本，避免后续单元覆盖变量
train_files_c1 = list(c1_train_files)
val_files_c1 = list(c1_val_files)
test_files_c1 = list(c1_test_files)

assert len(train_files_c1) == len(set(train_files_c1)) == 216
assert len(val_files_c1) == len(set(val_files_c1)) == 54
assert len(test_files_c1) == len(set(test_files_c1)) == 129
assert set(train_files_c1).isdisjoint(val_files_c1)


def c1_slide_id(filename):
    return Path(filename).stem.rsplit("_", 2)[0]


# 核对训练和验证切片身份、顺序均与A0对应
assert [c1_slide_id(f) for f in train_files_c1] == [
    c1_slide_id(f) for f in a0_split["train"]
]
assert [c1_slide_id(f) for f in val_files_c1] == [
    c1_slide_id(f) for f in a0_split["val"]
]

for folder, filenames in [
    (train_dir_c1, train_files_c1 + val_files_c1),
    (test_dir_c1, test_files_c1),
]:
    for filename in filenames:
        if not (folder / filename).is_file():
            raise FileNotFoundError(folder / filename)

c1_save_path = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/"
    "Ablation_NEW/CAMELYON16/C1_Level2_Joint_patience10/"
)
c1_save_path.mkdir(parents=True, exist_ok=True)

for filename in (
    "model.pth",
    "config.json",
    "split_on_CAMELYON16.json",
    "log.log",
    "training_summary.json",
):
    if (c1_save_path / filename).exists():
        raise FileExistsError(
            f"已有实验文件，请先检查或备份：{c1_save_path / filename}"
        )


class C1Level2Model(nn.Module):
    def __init__(self, gnn, mil):
        super().__init__()
        self.gnn = gnn
        self.mil = mil

    def forward(self, features, edges):
        return self.mil(self.gnn(features, edges))


def get_c1_data(path, device):
    # 核对实际元数据，防止误读其他尺度
    data = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )
    assert int(data["level"]) == 2, (
        f"文件不是Level 2：{path}"
    )
    del data

    # 沿用相同八邻域构图方式，步长取该文件patch_size
    return get_a2_data(path, device)


# 与A0保持相同构造顺序和默认参数
model_gnn = GNNModel(384, 256, 256, num_layers=1)
model_iclassifier = FCLayer(256, 2)
model_bclassifier = BClassifier(256, 2)
model_mil = MILNet(model_iclassifier, model_bclassifier)
model = C1Level2Model(model_gnn, model_mil).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4
)

config = {
    "experiment": "C1_Level2_Joint_patience10",
    "dataset": "CAMELYON16",
    "scale_folder": "10xbags",
    "wsi_level": 2,
    "feature_dim": 384,
    "class_num": 2,
    "data_path": str(train_dir_c1),
    "test_data_path": str(test_dir_c1),
    "a0_split_source": str(split_source),
    "split_matching": "slide identity; preserve A0 train/val order",
    "graph_encoder": "GNNModel(384,256,256,num_layers=1)",
    "graph_edges": "eight-neighbor; internal GAT self-loops",
    "aggregation_head": "original BClassifier(256,2)",
    "training_loss": "0.5 bag CE + 0.5 max-instance CE",
    "optimizer": "Adam",
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "patience": 10,
    "early_stopping_logic": (
        "reset to 10 on improvement, then decrement every epoch"
    ),
    "consecutive_non_improvements_to_stop": 9,
    "checkpoint_selection": "minimum validation bag CE",
    "training_seed": None,
    "train_order": "fixed mapped A0 order; no reshuffling",
    "encoder_checkpoint_provenance": "requires extraction records",
}

mapped_split = {
    "data_path": str(train_dir_c1),
    "test_data_path": str(test_dir_c1),
    "a0_split_source": str(split_source),
    "train": train_files_c1,
    "val": val_files_c1,
    "test": test_files_c1,
}

for filename, content in (
    ("config.json", config),
    ("split_on_CAMELYON16.json", mapped_split),
):
    with (c1_save_path / filename).open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

logger = logging.getLogger("C1_Level2_CAMELYON16")
logger.setLevel(logging.INFO)
logger.propagate = False

for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

handler = logging.FileHandler(
    c1_save_path / "log.log",
    mode="a",
    encoding="utf-8",
)
handler.setFormatter(
    logging.Formatter("%(asctime)s - %(message)s")
)
logger.addHandler(handler)
logger.info(json.dumps(config, ensure_ascii=False))

print("实验：C1，Level 2＋GAT1＋原聚合器＋联合CE", flush=True)
print("训练特征：", train_dir_c1, flush=True)
print("保存目录：", c1_save_path, flush=True)

best_val_ce = float("inf")
best_epoch = None
remaining = 10

try:
    for epoch in range(100):
        model.train()
        train_losses = []

        for filename in tqdm(
            train_files_c1,
            desc=f"C1 Level2 Epoch {epoch} Train",
        ):
            features, edges, target = get_c1_data(
                train_dir_c1 / filename, device
            )
            optimizer.zero_grad()

            instance_logits, bag_logits, _, _ = model(
                features, edges
            )

            assert tuple(bag_logits.shape) == (1, 2)
            assert tuple(instance_logits.shape) == (
                features.shape[0], 2
            )

            max_logits = instance_logits.max(
                dim=0, keepdim=True
            ).values

            loss = (
                0.5 * F.cross_entropy(bag_logits, target)
                + 0.5 * F.cross_entropy(max_logits, target)
            )

            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"非有限训练损失：{filename}"
                )

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for filename in tqdm(
                val_files_c1,
                desc=f"C1 Level2 Epoch {epoch} Val",
            ):
                features, edges, target = get_c1_data(
                    train_dir_c1 / filename, device
                )
                _, bag_logits, _, _ = model(features, edges)
                loss = F.cross_entropy(bag_logits, target)

                if not torch.isfinite(loss).item():
                    raise FloatingPointError(
                        f"非有限验证损失：{filename}"
                    )

                val_losses.append(loss.item())

        train_ce = float(np.mean(train_losses))
        val_ce = float(np.mean(val_losses))
        improved = val_ce < best_val_ce

        if improved:
            best_val_ce = val_ce
            best_epoch = epoch
            remaining = 10
            torch.save(model, c1_save_path / "model.pth")

        remaining -= 1

        info = (
            f"epoch:{epoch}\tremaining:{remaining}\t"
            f"train:{train_ce:.6f}\tval:{val_ce:.6f}\t"
            + ("saved" if improved else "")
        )
        print(info, flush=True)
        logger.info(info)
        handler.flush()

        if remaining < 1:
            print("Early stopping", flush=True)
            logger.info("Early stopping")
            break

    summary = {
        "experiment": config["experiment"],
        "best_epoch": best_epoch,
        "best_validation_ce": best_val_ce,
        "weight_path": str(c1_save_path / "model.pth"),
    }

    with (c1_save_path / "training_summary.json").open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    logger.info("Training complete: %s", summary)
    print("\n训练完成：", summary)

finally:
    handler.flush()
    handler.close()
    logger.removeHandler(handler)

In [ ]:
# ============================================================
# C1尺度配置实验测试：CAMELYON16，10xbags / Level 2
# 模型：八邻域GAT1 + 原聚合器
# 测试切片：使用保存的跨尺度映射名单，共129张
# 指标：肿瘤类别1的Pre、Rec、F1及总体Acc
# ============================================================

from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

run_dir = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/"
    "Ablation_NEW/CAMELYON16/C1_Level2_Joint_patience10/"
)

required = [
    "C1Level2Model",
    "get_c1_data",
    "get_a2_data",
    "GNNModel",
    "FCLayer",
    "BClassifier",
    "MILNet",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"请先恢复这些模型及函数定义：{missing}")

for filename in (
    "model.pth",
    "config.json",
    "split_on_CAMELYON16.json",
    "training_summary.json",
):
    if not (run_dir / filename).is_file():
        raise FileNotFoundError(run_dir / filename)

test_config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)
assert test_config["experiment"] == "C1_Level2_Joint_patience10"

saved_split = json.loads(
    (run_dir / "split_on_CAMELYON16.json").read_text(
        encoding="utf-8"
    )
)
test_dir = Path(saved_split["test_data_path"])
test_files = saved_split["test"]

assert len(test_files) == len(set(test_files)) == 129

model = torch.load(
    run_dir / "model.pth",
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

labels, predictions, rows = [], [], []

with torch.no_grad():
    for filename in tqdm(test_files, desc="C1 Level2 Test"):
        path = test_dir / filename
        features, edges, target = get_c1_data(path, device)

        _, bag_logits, _, _ = model(features, edges)

        assert tuple(bag_logits.shape) == (1, 2)
        assert torch.isfinite(bag_logits).all()

        true_label = int(target.item())
        prediction = int(bag_logits.argmax(dim=1).item())

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": filename,
            "slide_id": Path(filename).stem.rsplit("_", 2)[0],
            "label": true_label,
            "prediction": prediction,
            "patch_count": int(features.shape[0]),
            "tumor_probability": float(
                bag_logits.softmax(dim=1)[0, 1].item()
            ),
        })

assert [labels.count(i) for i in (0, 1)] == [81, 48]

result = {
    "model": "CAMELYON16_C1_Level2_Joint_patience10",
    "scale_folder": "10xbags",
    "wsi_level": 2,
    "weight_path": str((run_dir / "model.pth").resolve()),
    "test_data_path": str(test_dir.resolve()),
    "n": len(labels),
    "average": "binary",
    "positive_label": 1,
    "precision": float(precision_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "accuracy": float(accuracy_score(labels, predictions)),
    "recall": float(recall_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "confusion_matrix_labels_0_1": confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别；顺序0、1")
print(np.array(result["confusion_matrix_labels_0_1"]))

for filename, content in (
    ("test_metrics.json", result),
    ("test_predictions.json", rows),
):
    with (run_dir / filename).open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

log_path = run_dir / "log.log"
with log_path.open("a", encoding="utf-8") as f:
    f.write("\n" + summary + "\n")
    f.write(
        "Confusion matrix [0, 1]:\n"
        + str(result["confusion_matrix_labels_0_1"])
        + "\n"
    )

assert summary in log_path.read_text(encoding="utf-8")
print("测试结果和日志已保存至：", run_dir.resolve())

In [ ]:
# ============================================================
# C2 尺度配置实验：CAMELYON16，5xbags / Level 3
# 模型：八邻域GAT1 + 原关键实例聚合器
# 监督：0.5包级CE + 0.5最大实例CE
# 划分：按切片ID沿用A0，216训练 / 54验证 / 129测试
# 本单元：核对特征、映射划分、定义模型，不开始训练
# ============================================================

import json
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

from wsi.nets.dasmil import (
    GNNModel,
    FCLayer,
    BClassifier,
    MILNet,
)

if "get_a2_data" not in globals():
    raise NameError("请先运行上一条提供的get_a2_data定义单元。")

project_dir_c2 = Path("/data/code/WSI/WSI_pretrain")
base_dir_c2 = Path("/data/database/CAMELYON16/pkl_data")
a0_dir_c2 = (
    project_dir_c2
    / "model/MIL/compare/Camelyon/Ours/Ours_patience10"
)

split_candidates_c2 = [
    a0_dir_c2 / "split_on_CAMELYON16.json",
    a0_dir_c2 / "split.json",
    project_dir_c2 / (
        "model/MIL/compare/Camelyon/"
        "patience10_split_on_CAMELYON16.json"
    ),
    project_dir_c2 / (
        "model/MIL/compare/Camelyon/patience10_split.json"
    ),
]

c2_split_source = next(
    (p for p in split_candidates_c2 if p.is_file()),
    None,
)
if c2_split_source is None:
    raise FileNotFoundError("没有找到A0或公共划分。")

a0_split_c2 = json.loads(
    c2_split_source.read_text(encoding="utf-8")
)


def find_c2_dir(subset):
    candidates = [
        base_dir_c2 / subset / "5xbags",
        base_dir_c2 / subset / "5",
    ]
    found = [p for p in candidates if p.is_dir()]
    if len(found) != 1:
        raise RuntimeError(
            f"{subset}的5尺度目录应唯一，当前找到：{found}"
        )
    return found[0]


c2_train_dir = find_c2_dir("training")
c2_test_dir = find_c2_dir("testing")


def parse_c2_name(filename):
    slide_id, patch_count, label = (
        Path(filename).stem.rsplit("_", 2)
    )
    return slide_id, int(patch_count), int(label)


def build_c2_index(folder):
    index = {}
    for path in sorted(folder.glob("*.pth")):
        slide_id, _, _ = parse_c2_name(path.name)
        if slide_id in index:
            raise ValueError(f"重复切片ID：{folder}/{slide_id}")
        index[slide_id] = path.name
    return index


def map_c2_files(original_names, target_index):
    mapped = []
    for name in original_names:
        slide_id, _, original_label = parse_c2_name(name)

        if slide_id not in target_index:
            raise FileNotFoundError(f"5尺度缺少切片：{slide_id}")

        target_name = target_index[slide_id]
        _, _, target_label = parse_c2_name(target_name)

        if original_label != target_label:
            raise ValueError(f"跨尺度标签不一致：{slide_id}")

        mapped.append(target_name)
    return mapped


c2_train_index = build_c2_index(c2_train_dir)
c2_test_index = build_c2_index(c2_test_dir)

c2_train_files = map_c2_files(
    a0_split_c2["train"], c2_train_index
)
c2_val_files = map_c2_files(
    a0_split_c2["val"], c2_train_index
)

reference_test_dir_c2 = base_dir_c2 / "testing/20xbags"
reference_test_files_c2 = sorted(
    p.name for p in reference_test_dir_c2.glob("*.pth")
    if p.is_file()
)
assert len(reference_test_files_c2) == 129

c2_test_files = map_c2_files(
    reference_test_files_c2, c2_test_index
)

assert len(c2_train_files) == len(set(c2_train_files)) == 216
assert len(c2_val_files) == len(set(c2_val_files)) == 54
assert len(c2_test_files) == len(set(c2_test_files)) == 129
assert set(c2_train_files).isdisjoint(c2_val_files)

assert set(c2_train_files + c2_val_files) == set(
    c2_train_index.values()
)
assert set(c2_test_files) == set(c2_test_index.values())


def get_c2_data(path, device):
    # 与C1一致，先核对实际尺度元数据
    data = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )
    if int(data["level"]) != 3:
        raise ValueError(
            f"预期Level 3，实际为{data['level']}：{path}"
        )
    del data

    return get_a2_data(path, device)


class C2Level3Model(nn.Module):
    def __init__(self, gnn, mil):
        super().__init__()
        self.gnn = gnn
        self.mil = mil

    def forward(self, features, edges):
        return self.mil(self.gnn(features, edges))


# 先抽查，发现尺度不符时在训练前停止
for folder, filename in [
    (c2_train_dir, c2_train_files[0]),
    (c2_test_dir, c2_test_files[0]),
]:
    features, edges, label = get_c2_data(
        folder / filename, "cpu"
    )
    print({
        "file": filename,
        "feature_shape": tuple(features.shape),
        "edge_shape": tuple(edges.shape),
        "label": int(label.item()),
        "level": 3,
    })

print("A0划分：", c2_split_source)
print("C2训练目录：", c2_train_dir)
print("C2测试目录：", c2_test_dir)
print("训练 / 验证 / 测试：216 / 54 / 129")
print("C2准备完成，未重新随机划分。")

In [ ]:
# ============================================================
# C2训练：Level 3 + 八邻域GAT1 + 原聚合器 + 联合CE
# 优化：Adam，lr=3e-4，最多100轮，patience=10
# 选模：最低验证包级CE
# 保存：Ablation_NEW/CAMELYON16/C2_Level3_Joint_patience10
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA不可用，请检查GPU和Notebook内核。")

device = torch.device("cuda:0")

c2_save_path = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/"
    "Ablation_NEW/CAMELYON16/C2_Level3_Joint_patience10/"
)
c2_save_path.mkdir(parents=True, exist_ok=True)

for filename in (
    "model.pth",
    "config.json",
    "split_on_CAMELYON16.json",
    "log.log",
    "training_summary.json",
):
    if (c2_save_path / filename).exists():
        raise FileExistsError(
            f"已有实验文件，请先检查或备份：{c2_save_path / filename}"
        )

# 与A0/C1保持相同模型构造顺序及默认参数
model_gnn = GNNModel(384, 256, 256, num_layers=1)
model_iclassifier = FCLayer(256, 2)
model_bclassifier = BClassifier(256, 2)
model_mil = MILNet(model_iclassifier, model_bclassifier)
model = C2Level3Model(model_gnn, model_mil).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=3e-4
)

config = {
    "experiment": "C2_Level3_Joint_patience10",
    "dataset": "CAMELYON16",
    "scale_folder": c2_train_dir.name,
    "wsi_level": 3,
    "feature_dim": 384,
    "class_num": 2,
    "data_path": str(c2_train_dir),
    "test_data_path": str(c2_test_dir),
    "a0_split_source": str(c2_split_source),
    "split_matching": "slide identity; preserve A0 train/val order",
    "graph_encoder": "GNNModel(384,256,256,num_layers=1)",
    "graph_edges": "eight-neighbor; internal GAT self-loops",
    "aggregation_head": "original BClassifier(256,2)",
    "training_loss": "0.5 bag CE + 0.5 max-instance CE",
    "optimizer": "Adam",
    "learning_rate": 3e-4,
    "max_epochs": 100,
    "patience": 10,
    "early_stopping_logic": (
        "reset to 10 on improvement, then decrement every epoch"
    ),
    "consecutive_non_improvements_to_stop": 9,
    "checkpoint_selection": "minimum validation bag CE",
    "training_seed": None,
    "train_order": "fixed mapped A0 order; no reshuffling",
    "encoder_checkpoint_provenance": "requires extraction records",
}

mapped_split = {
    "data_path": str(c2_train_dir),
    "test_data_path": str(c2_test_dir),
    "a0_split_source": str(c2_split_source),
    "train": list(c2_train_files),
    "val": list(c2_val_files),
    "test": list(c2_test_files),
}

for filename, content in (
    ("config.json", config),
    ("split_on_CAMELYON16.json", mapped_split),
):
    with (c2_save_path / filename).open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

logger = logging.getLogger("C2_Level3_CAMELYON16")
logger.setLevel(logging.INFO)
logger.propagate = False

for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

handler = logging.FileHandler(
    c2_save_path / "log.log",
    mode="a",
    encoding="utf-8",
)
handler.setFormatter(
    logging.Formatter("%(asctime)s - %(message)s")
)
logger.addHandler(handler)
logger.info(json.dumps(config, ensure_ascii=False))

print("实验：C2，Level 3＋GAT1＋原聚合器＋联合CE", flush=True)
print("保存目录：", c2_save_path, flush=True)

best_val_ce = float("inf")
best_epoch = None
remaining = 10

try:
    for epoch in range(100):
        model.train()
        train_losses = []

        for filename in tqdm(
            c2_train_files,
            desc=f"C2 Level3 Epoch {epoch} Train",
        ):
            features, edges, target = get_c2_data(
                c2_train_dir / filename, device
            )
            optimizer.zero_grad()

            instance_logits, bag_logits, _, _ = model(
                features, edges
            )

            assert tuple(bag_logits.shape) == (1, 2)
            assert tuple(instance_logits.shape) == (
                features.shape[0], 2
            )

            max_logits = instance_logits.max(
                dim=0, keepdim=True
            ).values

            loss = (
                0.5 * F.cross_entropy(bag_logits, target)
                + 0.5 * F.cross_entropy(max_logits, target)
            )

            if not torch.isfinite(loss).item():
                raise FloatingPointError(
                    f"非有限训练损失：{filename}"
                )

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for filename in tqdm(
                c2_val_files,
                desc=f"C2 Level3 Epoch {epoch} Val",
            ):
                features, edges, target = get_c2_data(
                    c2_train_dir / filename, device
                )
                _, bag_logits, _, _ = model(features, edges)
                loss = F.cross_entropy(bag_logits, target)

                if not torch.isfinite(loss).item():
                    raise FloatingPointError(
                        f"非有限验证损失：{filename}"
                    )

                val_losses.append(loss.item())

        train_ce = float(np.mean(train_losses))
        val_ce = float(np.mean(val_losses))
        improved = val_ce < best_val_ce

        if improved:
            best_val_ce = val_ce
            best_epoch = epoch
            remaining = 10
            torch.save(model, c2_save_path / "model.pth")

        remaining -= 1

        info = (
            f"epoch:{epoch}\tremaining:{remaining}\t"
            f"train:{train_ce:.6f}\tval:{val_ce:.6f}\t"
            + ("saved" if improved else "")
        )
        print(info, flush=True)
        logger.info(info)
        handler.flush()

        if remaining < 1:
            print("Early stopping", flush=True)
            logger.info("Early stopping")
            break

    summary = {
        "experiment": config["experiment"],
        "best_epoch": best_epoch,
        "best_validation_ce": best_val_ce,
        "weight_path": str(c2_save_path / "model.pth"),
    }

    with (c2_save_path / "training_summary.json").open(
        "x", encoding="utf-8"
    ) as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    logger.info("Training complete: %s", summary)
    print("\n训练完成：", summary)

finally:
    handler.flush()
    handler.close()
    logger.removeHandler(handler)

In [ ]:
# ============================================================
# C2尺度配置实验测试：CAMELYON16，5xbags / Level 3
# 模型：八邻域GAT1 + 原聚合器
# 测试：保存的129张跨尺度对应切片
# 指标：肿瘤类别1的Pre、Rec、F1及总体Acc
# ============================================================

from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

run_dir = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/"
    "Ablation_NEW/CAMELYON16/C2_Level3_Joint_patience10/"
)

required = [
    "C2Level3Model",
    "get_c2_data",
    "get_a2_data",
    "GNNModel",
    "FCLayer",
    "BClassifier",
    "MILNet",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"请先恢复这些类和函数定义：{missing}")

for filename in (
    "model.pth",
    "config.json",
    "split_on_CAMELYON16.json",
    "training_summary.json",
):
    if not (run_dir / filename).is_file():
        raise FileNotFoundError(run_dir / filename)

test_config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)
assert test_config["experiment"] == "C2_Level3_Joint_patience10"

saved_split = json.loads(
    (run_dir / "split_on_CAMELYON16.json").read_text(
        encoding="utf-8"
    )
)
test_dir = Path(saved_split["test_data_path"])
test_files = saved_split["test"]

assert len(test_files) == len(set(test_files)) == 129

model = torch.load(
    run_dir / "model.pth",
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

labels, predictions, rows = [], [], []

with torch.no_grad():
    for filename in tqdm(test_files, desc="C2 Level3 Test"):
        path = test_dir / filename
        features, edges, target = get_c2_data(path, device)

        _, bag_logits, _, _ = model(features, edges)

        assert tuple(bag_logits.shape) == (1, 2)
        assert torch.isfinite(bag_logits).all()

        true_label = int(target.item())
        prediction = int(bag_logits.argmax(dim=1).item())

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": filename,
            "slide_id": Path(filename).stem.rsplit("_", 2)[0],
            "label": true_label,
            "prediction": prediction,
            "patch_count": int(features.shape[0]),
            "tumor_probability": float(
                bag_logits.softmax(dim=1)[0, 1].item()
            ),
        })

assert [labels.count(i) for i in (0, 1)] == [81, 48]

result = {
    "model": "CAMELYON16_C2_Level3_Joint_patience10",
    "scale_folder": test_config["scale_folder"],
    "wsi_level": 3,
    "weight_path": str((run_dir / "model.pth").resolve()),
    "test_data_path": str(test_dir.resolve()),
    "n": len(labels),
    "average": "binary",
    "positive_label": 1,
    "precision": float(precision_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "accuracy": float(accuracy_score(labels, predictions)),
    "recall": float(recall_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions, pos_label=1, zero_division=0,
    )),
    "confusion_matrix_labels_0_1": confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别；顺序0、1")
print(np.array(result["confusion_matrix_labels_0_1"]))

for filename, content in (
    ("test_metrics.json", result),
    ("test_predictions.json", rows),
):
    with (run_dir / filename).open(
        "w", encoding="utf-8"
    ) as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

log_path = run_dir / "log.log"
with log_path.open("a", encoding="utf-8") as f:
    f.write("\n" + summary + "\n")
    f.write(
        "Confusion matrix [0, 1]:\n"
        + str(result["confusion_matrix_labels_0_1"])
        + "\n"
    )

assert summary in log_path.read_text(encoding="utf-8")
print("测试结果和日志已保存至：", run_dir.resolve())